In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
import pickle

In [2]:
# Creating MEASUREMENTS table
MEASUREMENTS = pd.DataFrame(columns = ['ROW_ID',
                                       'MEASUREMENTS_ID',
                                       'ADMISSION_ID',
                                       'PATIENT_ID',
                                       'ITEM_ID',
                                       'DATETIME',
                                       'VALUE_NUM',
                                       'VALUE_UOM',
                                       'VALUE_CHAR',
                                       'VALUE_FLAG'])

In [3]:
# Creating ITEMS table
ITEMS = pd.DataFrame(columns = ['ROW_ID',
                                'ITEM_ID',
                                'LABEL',
                                'ABBREVIATION',
                                'CATEGORY',
                                'PARAM_TYPE',
                                'UOM'])

In [4]:
# Opening ADMISSIONS for ADMISSION_ID, PATIENT_ID, RECORD_ID reference
ADMISSIONS = pd.read_csv('ADMISSIONS.csv', delimiter = ";", encoding = 'Latin-1')
ADMISSIONS = ADMISSIONS[['ADMISSION_ID', 'PATIENT_ID', 'RECORD_ID']]
ADMISSIONS.head()

,ADMISSION_ID,PATIENT_ID,RECORD_ID
0,A00000,P00033,189844
1,A00001,P00005,189946
2,A00002,P00033,190055
3,A00003,P00175,190072
4,A00004,P00010,190106


In [5]:
icu_file_path = '/home/lu.kuang/thesis/Prospective_DataCleaning/updated_cleaning/DataMerged_Prospective_0430_updated.xlsx'

## 1) ADD GCS

In [6]:
# preparing GCS for inclusion
GCS = pd.read_excel(io = icu_file_path, sheet_name = 'GCS')
GCS

,RECORD_ID,SITE,MRN,ICU_ADMIT,parameter,chartdate,value_num,value_char
0,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Eye Opening GCS,2021-08-01 02:14:00,NaN,Spontaneously
1,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Motor Response GCS,2021-08-01 02:14:00,NaN,Obeys Commands
2,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Verbal Response GCS,2021-08-01 02:14:00,NaN,Oriented
3,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Best Eye Opening GCS,2021-08-01 04:30:00,NaN,Opens to Verbal Command
4,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Best Motor Response GCS,2021-08-01 04:30:00,NaN,Obeys
...,...,...,...,...,...,...,...,...
79867,207475,FMC ICU,1045503818,2022-05-26 00:34:00,Best Verbal Response Unable to Assess Reason GCS,2022-05-29 08:00:00,NaN,Intubated or Tracheostomy
79868,207475,FMC ICU,1045503818,2022-05-26 00:34:00,Eye Opening GCS,2022-05-29 08:00:00,NaN,No Response
79869,207475,FMC ICU,1045503818,2022-05-26 00:34:00,GCS RN,2022-05-29 08:00:00,3.0,NaN
79870,207475,FMC ICU,1045503818,2022-05-26 00:34:00,Motor Response GCS,2022-05-29 08:00:00,NaN,No Response


In [7]:
# merging data and ADMISSIONS
GCS = GCS.merge(right = ADMISSIONS, on = ['RECORD_ID'])
GCS.drop(['RECORD_ID'], axis = 1, inplace = True)
GCS.sort_values(by = ['ADMISSION_ID','chartdate'], inplace = True)

In [8]:
GCS

,SITE,MRN,ICU_ADMIT,parameter,chartdate,value_num,value_char,ADMISSION_ID,PATIENT_ID
0,FMC ICU,1005018831,2021-08-01 02:14:00,Eye Opening GCS,2021-08-01 02:14:00,NaN,Spontaneously,A00000,P00033
1,FMC ICU,1005018831,2021-08-01 02:14:00,Motor Response GCS,2021-08-01 02:14:00,NaN,Obeys Commands,A00000,P00033
2,FMC ICU,1005018831,2021-08-01 02:14:00,Verbal Response GCS,2021-08-01 02:14:00,NaN,Oriented,A00000,P00033
3,FMC ICU,1005018831,2021-08-01 02:14:00,Best Eye Opening GCS,2021-08-01 04:30:00,NaN,Opens to Verbal Command,A00000,P00033
4,FMC ICU,1005018831,2021-08-01 02:14:00,Best Motor Response GCS,2021-08-01 04:30:00,NaN,Obeys,A00000,P00033
...,...,...,...,...,...,...,...,...,...
75833,FMC ICU,1021916406,2022-07-22 21:03:00,Best Verbal Response Unable to Assess Reason GCS,2022-07-26 05:00:00,NaN,Intubated or Tracheostomy,A00351,P00118
75835,FMC ICU,1021916406,2022-07-22 21:03:00,GCS RN,2022-07-26 05:00:00,3.0,NaN,A00351,P00118
75837,FMC ICU,1021916406,2022-07-22 21:03:00,Eye Opening GCS,2022-07-26 08:00:00,NaN,No Response,A00351,P00118
75839,FMC ICU,1021916406,2022-07-22 21:03:00,Motor Response GCS,2022-07-26 08:00:00,NaN,No Response,A00351,P00118


In [ ]:
# Mapping dictionary for GCS values
gcs_mapping = {
    # Eye Opening
    'Opens Spontaneously': 4, 
    'Spontaneously': 4,
    'Opens to Verbal Command': 3, 
    'To Speech': 3,
    'Opens to Pain': 2,    
    'To Pain': 2,
    
    # Verbal Response
    'Oriented': 5, 
    'Confused': 4, 
    'Inappropriate Words': 3,
    'Incomprehensible Sounds': 2,    
    'Incomprehensible': 2,
    
    # Motor Response
    'Obeys Commands': 6, 
    'Obeys': 6,
    'Localizes to Pain': 5, 
    'Localizes': 5,
    'Withdrawal from Pain': 4,
    'Withdrawal': 4,
    'Abnormal Flexion': 3, 
    'Abnormal Extension': 2,    
    
    # Shared by all
    'Unable to Assess': 1,
    'No Response': 1
}

# Replace values using loc to avoid the warning
GCS.loc[:, 'value_char'] = GCS['value_char'].replace(gcs_mapping)

/tmp/ipykernel_3678869/3435852454.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  GCS['value_char'].replace({


In [ ]:
# Mapping dictionary for GCS parameters
gcs_parameter_mapping = {
    'GCS RN': 'GCS',
    
    'Motor Response GCS': 'Motor Response',
    'Eye Opening GCS': 'Eye Opening', 
    'Verbal Response GCS': 'Verbal Response',
    
    'Best Motor Response GCS': 'Motor Response',
    'Best Eye Opening GCS': 'Eye Opening',
    'Best Verbal Response GCS': 'Verbal Response',
    
    'Best Eye Opening Unable to Assess Reason GCS': 'Eye Opening - Unable to Assess',
    'Best Motor Response Unable to Assess Reason GCS': 'Motor Response - Unable to Assess', 
    'Best Verbal Response Unable to Assess Reason GCS': 'Verbal Response - Unable to Assess'
}

# Replace values using loc to avoid the warning
GCS.loc[:, 'parameter'] = GCS['parameter'].replace(gcs_parameter_mapping)

/tmp/ipykernel_3678869/1217259697.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  GCS['parameter'].replace({


In [11]:
# copying values to value_num
GCS['value_num'] = GCS.apply(lambda x: 
                                 
                               float(x['value_char'])
                               if 
                               (x['parameter'] == 'Eye Opening') | 
                               (x['parameter'] == 'Motor Response') | 
                               (x['parameter'] == 'Verbal Response') 
                               else                                     
                               x['value_num'], 
                                    
                               axis = 1)

In [12]:
# adjusting value_char
GCS['value_char'] = GCS.apply(lambda x: 
                                    
                                np.nan                                     
                                if 
                                (x['parameter'] == 'Eye Opening') | 
                                (x['parameter'] == 'Motor Response') | 
                                (x['parameter'] == 'Verbal Response') 
                                else                                     
                                x['value_char'], 

                                axis = 1)

In [13]:
temp = GCS[['ADMISSION_ID','chartdate','value_num']].groupby(['ADMISSION_ID','chartdate']).sum()
temp['MEASUREMENTS_ID'] = range(0,len(temp))
temp['MEASUREMENTS_ID'] = temp['MEASUREMENTS_ID'].apply(lambda x: 'M'+str(x).rjust(8, '0'))
temp.reset_index(inplace = True)
temp.rename(columns = {'value_num' : 'temp_gcs', 'parameter' : 'temp_parameter'}, inplace = True)
GCS = GCS.merge(right = temp, on = ['ADMISSION_ID','chartdate'])

In [14]:
GCS

,SITE,MRN,ICU_ADMIT,parameter,chartdate,value_num,value_char,ADMISSION_ID,PATIENT_ID,temp_gcs,MEASUREMENTS_ID
0,FMC ICU,1005018831,2021-08-01 02:14:00,Eye Opening,2021-08-01 02:14:00,4.0,NaN,A00000,P00033,15.0,M00000000
1,FMC ICU,1005018831,2021-08-01 02:14:00,Motor Response,2021-08-01 02:14:00,6.0,NaN,A00000,P00033,15.0,M00000000
2,FMC ICU,1005018831,2021-08-01 02:14:00,Verbal Response,2021-08-01 02:14:00,5.0,NaN,A00000,P00033,15.0,M00000000
3,FMC ICU,1005018831,2021-08-01 02:14:00,Eye Opening,2021-08-01 04:30:00,3.0,NaN,A00000,P00033,20.0,M00000001
4,FMC ICU,1005018831,2021-08-01 02:14:00,Motor Response,2021-08-01 04:30:00,6.0,NaN,A00000,P00033,20.0,M00000001
...,...,...,...,...,...,...,...,...,...,...,...
90452,FMC ICU,1021916406,2022-07-22 21:03:00,Verbal Response - Unable to Assess,2022-07-26 05:00:00,NaN,Intubated or Tracheostomy,A00351,P00118,6.0,M00014882
90453,FMC ICU,1021916406,2022-07-22 21:03:00,GCS,2022-07-26 05:00:00,3.0,NaN,A00351,P00118,6.0,M00014882
90454,FMC ICU,1021916406,2022-07-22 21:03:00,Eye Opening,2022-07-26 08:00:00,1.0,NaN,A00351,P00118,3.0,M00014883
90455,FMC ICU,1021916406,2022-07-22 21:03:00,Motor Response,2022-07-26 08:00:00,1.0,NaN,A00351,P00118,3.0,M00014883


In [15]:
# Calculating and adding GCS for MEASUREMENTS_ID that have 'Eye Opening', 'Motor Response' and 'Verbal Response', but no 'GCS'
# Concatenating parameters available for each MEASUREMENTS_ID
temp = GCS[['parameter','MEASUREMENTS_ID']].groupby('MEASUREMENTS_ID')['parameter'].apply(','.join).reset_index()

# flagging cases where Eye Opening', 'Motor Response' and 'Verbal Response' are available, but there is no 'GCS'
temp['flag'] = temp.apply(lambda x:
                          1
                          if
                          'Eye Opening' in x['parameter'] and
                          'Motor Response' in x['parameter'] and
                          'Verbal Response' in x['parameter'] and
                          'GCS' not in x['parameter']
                          else
                          0, 
                          axis = 1)

# Filtering data to retain only one row of each MEASUREMENTS_ID where flag == 1
temp = temp[temp['flag'] == 1].reset_index()
IDs = temp['MEASUREMENTS_ID'].unique()
temp = GCS[GCS['MEASUREMENTS_ID'].isin(IDs)]
temp.loc[:,'parameter'] = 'GCS'
temp.loc[:,'value_num'] = temp.loc[:,'temp_gcs']
temp.loc[:,'value_char'] = np.nan
temp = temp.drop_duplicates()

# appending temp to data
GCS = pd.concat([GCS, temp], axis = 0, ignore_index = True)
GCS.drop(['temp_gcs'], axis = 1, inplace = True)
GCS.sort_values(by = ['MEASUREMENTS_ID'], inplace = True)
GCS.reset_index(drop = True, inplace = True)

In [16]:
# preparing data to look like MEASUREMENTS table
GCS.rename(columns={'parameter' : 'ITEM_ID',
                     'chartdate' : 'DATETIME',
                     'value_num' : 'VALUE_NUM',
                     'value_char' : 'VALUE_CHAR',}, 
            inplace = True)

In [ ]:
# Prepare ITEM_ID info
def determine_param_type(item, df):
    """Determine parameter type based on first non-null numerical value"""
    num_value = df[df['ITEM_ID'] == item]['VALUE_NUM'].dropna().iloc[0] if len(df[df['ITEM_ID'] == item]['VALUE_NUM'].dropna()) > 0 else np.nan
    return 'NUM' if not np.isnan(num_value) else 'CHAR'

# Create initial dataframe with unique ITEM_IDs
temp = pd.DataFrame({
    'LABEL': GCS['ITEM_ID'].unique()
})

# Add remaining columns
temp['ROW_ID'] = range(len(temp))
temp['ITEM_ID'] = temp['ROW_ID'].apply(lambda x: f'I{str(x).zfill(3)}')
temp['ABBREVIATION'] = temp['LABEL']
temp['CATEGORY'] = 'GCS'
temp['PARAM_TYPE'] = temp['LABEL'].apply(lambda x: determine_param_type(x, GCS))

# Create mapping dictionary for original parameter labels
item_dict = temp.set_index('LABEL')['ITEM_ID'].to_dict()

# Update label for GCS
temp.loc[temp['LABEL'] == 'GCS', 'LABEL'] = 'Glasgow Coma Scale'

/tmp/ipykernel_3678869/1407116477.py:23: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  temp['LABEL'].replace({'GCS' : 'Glasgow Coma Scale'}, inplace = True)


In [ ]:
# Adjusting data using item_dict
GCS = (GCS
       # Add LABEL column and map ITEM_IDs
       .assign(
           LABEL=lambda x: x['ITEM_ID'],
           ITEM_ID=lambda x: x['ITEM_ID'].map(item_dict['ITEM_ID'])
       )
       # Sort values
       .sort_values(['MEASUREMENTS_ID', 'ITEM_ID'])
       .reset_index(drop=True)
)

/tmp/ipykernel_3678869/174308609.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  GCS['ITEM_ID'].replace(item_dict['ITEM_ID'], inplace = True)


In [ ]:
# Append new data to master tables
def append_to_tables(items_df, measurements_df, new_items, new_measurements):
    """Append new data to ITEMS and MEASUREMENTS tables"""
    # Concat ITEMS table
    updated_items = pd.concat(
        [items_df, new_items],
        axis=0, 
        ignore_index=True,
        verify_integrity=True
    )
    
    # Concat MEASUREMENTS table  
    updated_measurements = pd.concat(
        [measurements_df, new_measurements],
        axis=0,
        ignore_index=True, 
        verify_integrity=True
    )
    
    return updated_items, updated_measurements

# Append data
ITEMS_data, MEASUREMENTS_data = append_to_tables(ITEMS, temp, MEASUREMENTS, GCS)

/tmp/ipykernel_3678869/359375400.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  MEASUREMENTS_data = pd.concat([MEASUREMENTS,GCS], axis = 0, ignore_index = True)


In [20]:
# Trying to free some memory for the following steps
count_ref_measure = int(MEASUREMENTS_data['MEASUREMENTS_ID'].max()[1:]) + 1
count_ref_item = int(ITEMS_data['ITEM_ID'].max()[1:]) + 1
MEASUREMENTS_data.to_pickle('MEASUREMENTS_01.pickle')
ITEMS_data.to_pickle('ITEMS_01.pickle')

## 2) ADD ICDSC

In [21]:
# preparing ICDSC for inclusion
ICDSC = pd.read_excel(io = icu_file_path, sheet_name = 'ICDSC')
ICDSC

,RECORD_ID,SITE,MRN,ICU_ADMIT,parameter,chartdate,enddate,value_num,value_num_unit,value_char
0,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Disorientation,2021-08-01 04:00:00,2021-08-01 04:00:00,0.0,NaN,0
1,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Fluctuations,2021-08-01 04:00:00,2021-08-01 04:00:00,0.0,NaN,0
2,189844,FMC ICU,1005018831,2021-08-01 02:14:00,"Hallucination, Delusion or Psychosis",2021-08-01 04:00:00,2021-08-01 04:00:00,0.0,NaN,0
3,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Altered Level of Consciousness,2021-08-01 04:00:00,2021-08-01 04:00:00,NaN,NaN,Responds to Moderate Stimulation (RASS = -3)
4,189844,FMC ICU,1005018831,2021-08-01 02:14:00,ICDSC Score,2021-08-01 04:00:00,2021-08-01 04:00:00,2.0,NaN,2
...,...,...,...,...,...,...,...,...,...,...
28643,209883,FMC ICU,1021916406,2022-07-22 21:03:00,Altered Level of Consciousness,2022-07-23 04:00:00,2022-07-23 04:00:00,NaN,NaN,Responds to Intense Stimulation or No Response...
28644,209883,FMC ICU,1021916406,2022-07-22 21:03:00,Altered Level of Consciousness,2022-07-23 16:00:00,2022-07-23 16:00:00,NaN,NaN,Responds to Intense Stimulation or No Response...
28645,209883,FMC ICU,1021916406,2022-07-22 21:03:00,Altered Level of Consciousness,2022-07-24 04:00:00,2022-07-24 04:00:00,NaN,NaN,Responds to Intense Stimulation or No Response...
28646,209883,FMC ICU,1021916406,2022-07-22 21:03:00,Altered Level of Consciousness,2022-07-25 04:00:00,2022-07-25 04:00:00,NaN,NaN,Responds to Intense Stimulation or No Response...


In [22]:
# merging ICDSC data and ADMISSIONS
merged_data = ICDSC.merge(right = ADMISSIONS, on = ['RECORD_ID'])
merged_data.drop(['RECORD_ID'], axis = 1, inplace = True)
merged_data.sort_values(by = ['ADMISSION_ID','chartdate'], inplace = True)
merged_data

,SITE,MRN,ICU_ADMIT,parameter,chartdate,enddate,value_num,value_num_unit,value_char,ADMISSION_ID,PATIENT_ID
0,FMC ICU,1005018831,2021-08-01 02:14:00,Disorientation,2021-08-01 04:00:00,2021-08-01 04:00:00,0.0,NaN,0,A00000,P00033
1,FMC ICU,1005018831,2021-08-01 02:14:00,Fluctuations,2021-08-01 04:00:00,2021-08-01 04:00:00,0.0,NaN,0,A00000,P00033
2,FMC ICU,1005018831,2021-08-01 02:14:00,"Hallucination, Delusion or Psychosis",2021-08-01 04:00:00,2021-08-01 04:00:00,0.0,NaN,0,A00000,P00033
3,FMC ICU,1005018831,2021-08-01 02:14:00,Altered Level of Consciousness,2021-08-01 04:00:00,2021-08-01 04:00:00,NaN,NaN,Responds to Moderate Stimulation (RASS = -3),A00000,P00033
4,FMC ICU,1005018831,2021-08-01 02:14:00,ICDSC Score,2021-08-01 04:00:00,2021-08-01 04:00:00,2.0,NaN,2,A00000,P00033
...,...,...,...,...,...,...,...,...,...,...,...
35334,FMC ICU,1021916406,2022-07-22 21:03:00,Altered Level of Consciousness,2022-07-24 04:00:00,2022-07-24 04:00:00,NaN,NaN,Responds to Intense Stimulation or No Response...,A00351,P00118
26864,FMC ICU,1021916406,2022-07-22 21:03:00,Altered Level of Consciousness,2022-07-25 04:00:00,2022-07-25 04:00:00,NaN,NaN,Responds to Intense Stimulation or No Response...,A00351,P00118
35336,FMC ICU,1021916406,2022-07-22 21:03:00,Altered Level of Consciousness,2022-07-25 04:00:00,2022-07-25 04:00:00,NaN,NaN,Responds to Intense Stimulation or No Response...,A00351,P00118
26866,FMC ICU,1021916406,2022-07-22 21:03:00,Altered Level of Consciousness,2022-07-26 05:00:00,2022-07-26 05:00:00,NaN,NaN,Responds to Intense Stimulation or No Response...,A00351,P00118


In [23]:
# droping parameters that are not useful
# Old_ICDSC ST: presents a date. The majority of time, it is the same as in chartdate.
# Sleep: only one record in the whole dataset
merged_data = merged_data[(merged_data['parameter'] != 'Old_ICDSC ST') & (merged_data['parameter'] != 'Sleep')].reset_index(drop = True)
merged_data

,SITE,MRN,ICU_ADMIT,parameter,chartdate,enddate,value_num,value_num_unit,value_char,ADMISSION_ID,PATIENT_ID
0,FMC ICU,1005018831,2021-08-01 02:14:00,Disorientation,2021-08-01 04:00:00,2021-08-01 04:00:00,0.0,NaN,0,A00000,P00033
1,FMC ICU,1005018831,2021-08-01 02:14:00,Fluctuations,2021-08-01 04:00:00,2021-08-01 04:00:00,0.0,NaN,0,A00000,P00033
2,FMC ICU,1005018831,2021-08-01 02:14:00,"Hallucination, Delusion or Psychosis",2021-08-01 04:00:00,2021-08-01 04:00:00,0.0,NaN,0,A00000,P00033
3,FMC ICU,1005018831,2021-08-01 02:14:00,Altered Level of Consciousness,2021-08-01 04:00:00,2021-08-01 04:00:00,NaN,NaN,Responds to Moderate Stimulation (RASS = -3),A00000,P00033
4,FMC ICU,1005018831,2021-08-01 02:14:00,ICDSC Score,2021-08-01 04:00:00,2021-08-01 04:00:00,2.0,NaN,2,A00000,P00033
...,...,...,...,...,...,...,...,...,...,...,...
35334,FMC ICU,1021916406,2022-07-22 21:03:00,Altered Level of Consciousness,2022-07-24 04:00:00,2022-07-24 04:00:00,NaN,NaN,Responds to Intense Stimulation or No Response...,A00351,P00118
35335,FMC ICU,1021916406,2022-07-22 21:03:00,Altered Level of Consciousness,2022-07-25 04:00:00,2022-07-25 04:00:00,NaN,NaN,Responds to Intense Stimulation or No Response...,A00351,P00118
35336,FMC ICU,1021916406,2022-07-22 21:03:00,Altered Level of Consciousness,2022-07-25 04:00:00,2022-07-25 04:00:00,NaN,NaN,Responds to Intense Stimulation or No Response...,A00351,P00118
35337,FMC ICU,1021916406,2022-07-22 21:03:00,Altered Level of Consciousness,2022-07-26 05:00:00,2022-07-26 05:00:00,NaN,NaN,Responds to Intense Stimulation or No Response...,A00351,P00118


In [24]:
# Adjusting Parameters names and values
# Adjusting Value_char
merged_data['value_char'].replace({
    # OLD_Disoriented to
    'Oriented x 3' : 0,
    'Person' : 1,
    'Place' : 1,
    'Time' : 1,
    'Inconsistent' : 1,

    # OLD_Inattention
    'Attentive and Focused' : 0,
    'Difficulty Following Conversation' : 1,
    'Easily Distracted' : 1,
    'Difficulty Shifting Focus' : 1,

    # OLD_Psychomotor Agitation or Retardation
    'Relaxed and Cooperative' : 0,
    'Picking and Pulling at Lines' : 1,
    'Striking Out at Staff' : 1,
    'Attempting to Get Out of Bed' : 1,
    'Hypoactive or Psychomotor Slowing' : 1,

    # OLD_Fluctuations
    'No Fluctuations or Positive Fluctuations' : 0,
    'Negative Fluctuation of Any ICDSC Indicator' : 1,

    # OLD2_Hallucination Delusion or Psychosis
    'No Apparent Hallucinations Delusions or Psychosis' : 0,
    'Visual Hallucinations' : 1,
    'Auditory Hallucinations' : 1,
    'Tactile Hallucinations' : 1,
    'Delusion Present' : 1,
    'Psychosis Present' : 1,

    # OLD2_Inappropriate Conversational Speech or Mood
    'Appropriate Conversational Speech' : 0,
    'Inappropriate Conversational Speech' : 1,
    'Inappropriate Mood Related to Event' : 1,
    'Inappropriate Shouting or Profane Speech' : 1,
    'Inappropriate Crying' : 1,
    'Incoherent' : 1,

    # OLD2_Sleep Wake Cycle
    'Normal Sleep at Night - Greater Than 4 Consecutive Hours of Sleep' : 0,
    'Sleeping < 4 Consecutive Hours at Night' : 1,
    'Sleeping > 4 Cumulative Hours During The Day' : 1,
    'Frequent Wakefulness at Night Requiring Repeated Sedation to Settle' : 1,

    # Altered Level of Consciousness
    'Exaggerated Response to Normal Stimulation (RASS = +1 to +4)' : 1,
    'Normal Wakefulness (RASS = 0)' : 0,
    'Responds to Moderate Stimulation (RASS = -3)' : 1,
    'Responds to Mild Stimulation (RASS = -1 to -2)' : 1,
    'Responds to Intense Stimulation or No Response (RASS = -4 or -5)' : 1},
    
    inplace = True)
merged_data

/tmp/ipykernel_3678869/2665246484.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_data['value_char'].replace({
/tmp/ipykernel_3678869/2665246484.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged_data['value_char'].replace({


,SITE,MRN,ICU_ADMIT,parameter,chartdate,enddate,value_num,value_num_unit,value_char,ADMISSION_ID,PATIENT_ID
0,FMC ICU,1005018831,2021-08-01 02:14:00,Disorientation,2021-08-01 04:00:00,2021-08-01 04:00:00,0.0,NaN,0,A00000,P00033
1,FMC ICU,1005018831,2021-08-01 02:14:00,Fluctuations,2021-08-01 04:00:00,2021-08-01 04:00:00,0.0,NaN,0,A00000,P00033
2,FMC ICU,1005018831,2021-08-01 02:14:00,"Hallucination, Delusion or Psychosis",2021-08-01 04:00:00,2021-08-01 04:00:00,0.0,NaN,0,A00000,P00033
3,FMC ICU,1005018831,2021-08-01 02:14:00,Altered Level of Consciousness,2021-08-01 04:00:00,2021-08-01 04:00:00,NaN,NaN,1,A00000,P00033
4,FMC ICU,1005018831,2021-08-01 02:14:00,ICDSC Score,2021-08-01 04:00:00,2021-08-01 04:00:00,2.0,NaN,2,A00000,P00033
...,...,...,...,...,...,...,...,...,...,...,...
35334,FMC ICU,1021916406,2022-07-22 21:03:00,Altered Level of Consciousness,2022-07-24 04:00:00,2022-07-24 04:00:00,NaN,NaN,1,A00351,P00118
35335,FMC ICU,1021916406,2022-07-22 21:03:00,Altered Level of Consciousness,2022-07-25 04:00:00,2022-07-25 04:00:00,NaN,NaN,1,A00351,P00118
35336,FMC ICU,1021916406,2022-07-22 21:03:00,Altered Level of Consciousness,2022-07-25 04:00:00,2022-07-25 04:00:00,NaN,NaN,1,A00351,P00118
35337,FMC ICU,1021916406,2022-07-22 21:03:00,Altered Level of Consciousness,2022-07-26 05:00:00,2022-07-26 05:00:00,NaN,NaN,1,A00351,P00118


In [25]:
# copying values to value_num
merged_data['value_num'] = merged_data.apply(lambda x: 
                                 
                               float(x['value_char'])
                               if 
                               (x['parameter'] == 'OLD_Disoriented to') |
                               (x['parameter'] == 'OLD_Inattention') |
                               (x['parameter'] == 'OLD_Psychomotor Agitation or Retardation') |
                               (x['parameter'] == 'OLD_Fluctuations') |
                               (x['parameter'] == 'OLD2_Hallucination Delusion or Psychosis') |
                               (x['parameter'] == 'OLD2_Inappropriate Conversational Speech or Mood') |
                               (x['parameter'] == 'OLD2_Sleep Wake Cycle') |
                               (x['parameter'] == 'Altered Level of Consciousness') 
                               else                                     
                               x['value_num'], 
                                    
                               axis = 1)

In [26]:
# Adjusting Parameters names
merged_data['parameter'].replace({
    'ICDSC Score' : 'ICDSC',
    'OLD_Disoriented to' : 'Disorientation',
    'OLD_Inattention' : 'Inattention',
    'OLD_Psychomotor Agitation or Retardation' : 'Psychomotor Agitation or Retardation',
    'OLD_Fluctuations' : 'Fluctuations',
    'OLD2_Hallucination Delusion or Psychosis' : 'Hallucination, Delusion or Psychosis',
    'OLD2_Inappropriate Conversational Speech or Mood' : 'Inappropriate Mood or Speech',
    'OLD2_Sleep Wake Cycle' : 'Sleep Wake Cycle Disturbance',
    'Sleep Wake Cycle' : 'Sleep Wake Cycle Disturbance'},
    
    inplace = True)

/tmp/ipykernel_3678869/368042550.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_data['parameter'].replace({


In [27]:
# creating MEASUREMENTS_ID based on 'ADMISSION_ID','chartdate'
temp = merged_data[['ADMISSION_ID','chartdate','value_num']].groupby(['ADMISSION_ID','chartdate']).sum()
temp['MEASUREMENTS_ID'] = range(count_ref_measure, count_ref_measure + len(temp))
temp['MEASUREMENTS_ID'] = temp['MEASUREMENTS_ID'].apply(lambda x: 'M'+str(x).rjust(8, '0'))
temp.reset_index(inplace = True)
temp.rename(columns = {'value_num' : 'temp_icdsc'}, inplace = True)
merged_data = merged_data.merge(right = temp, on = ['ADMISSION_ID','chartdate'])
merged_data

,SITE,MRN,ICU_ADMIT,parameter,chartdate,enddate,value_num,value_num_unit,value_char,ADMISSION_ID,PATIENT_ID,temp_icdsc,MEASUREMENTS_ID
0,FMC ICU,1005018831,2021-08-01 02:14:00,Disorientation,2021-08-01 04:00:00,2021-08-01 04:00:00,0.0,NaN,0,A00000,P00033,4.0,M00014884
1,FMC ICU,1005018831,2021-08-01 02:14:00,Fluctuations,2021-08-01 04:00:00,2021-08-01 04:00:00,0.0,NaN,0,A00000,P00033,4.0,M00014884
2,FMC ICU,1005018831,2021-08-01 02:14:00,"Hallucination, Delusion or Psychosis",2021-08-01 04:00:00,2021-08-01 04:00:00,0.0,NaN,0,A00000,P00033,4.0,M00014884
3,FMC ICU,1005018831,2021-08-01 02:14:00,Altered Level of Consciousness,2021-08-01 04:00:00,2021-08-01 04:00:00,1.0,NaN,1,A00000,P00033,4.0,M00014884
4,FMC ICU,1005018831,2021-08-01 02:14:00,ICDSC,2021-08-01 04:00:00,2021-08-01 04:00:00,2.0,NaN,2,A00000,P00033,4.0,M00014884
...,...,...,...,...,...,...,...,...,...,...,...,...,...
35334,FMC ICU,1021916406,2022-07-22 21:03:00,Altered Level of Consciousness,2022-07-24 04:00:00,2022-07-24 04:00:00,1.0,NaN,1,A00351,P00118,2.0,M00019110
35335,FMC ICU,1021916406,2022-07-22 21:03:00,Altered Level of Consciousness,2022-07-25 04:00:00,2022-07-25 04:00:00,1.0,NaN,1,A00351,P00118,2.0,M00019111
35336,FMC ICU,1021916406,2022-07-22 21:03:00,Altered Level of Consciousness,2022-07-25 04:00:00,2022-07-25 04:00:00,1.0,NaN,1,A00351,P00118,2.0,M00019111
35337,FMC ICU,1021916406,2022-07-22 21:03:00,Altered Level of Consciousness,2022-07-26 05:00:00,2022-07-26 05:00:00,1.0,NaN,1,A00351,P00118,2.0,M00019112


In [28]:
# Calculating and adding ICDSC for MEASUREMENTS_ID that have all parameters, but no 'ICDSC'
# Concatenating parameters available for each MEASUREMENTS_ID
temp = merged_data[['parameter','MEASUREMENTS_ID']].groupby('MEASUREMENTS_ID')['parameter'].apply(','.join).reset_index()

# flagging cases where Eye Opening', 'Motor Response' and 'Verbal Response' are available, but there is no 'ICDSC'
temp['flag'] = temp.apply(lambda x:
                          1
                          if
                          'Disorientation' in x['parameter'] and
                          'Fluctuations' in x['parameter'] and
                          'Hallucination, Delusion or Psychosis' in x['parameter'] and
                          'Inappropriate Mood or Speech' in x['parameter'] and
                          'Inattention' in x['parameter'] and
                          'Sleep Wake Cycle Disturbance' in x['parameter'] and
                          'Psychomotor Agitation or Retardation' in x['parameter'] and
                          'Altered Level of Consciousness' in x['parameter'] and
                          'ICDSC' not in x['parameter']
                          else
                          0, 
                          axis = 1)

# Filtering data to retain only one row of each MEASUREMENTS_ID where flag == 1
temp = temp[temp['flag'] == 1].reset_index()
IDs = temp['MEASUREMENTS_ID'].unique()
temp = merged_data[merged_data['MEASUREMENTS_ID'].isin(IDs)]
temp.loc[:,'parameter'] = 'ICDSC'
temp.loc[:,'value_num'] = temp.loc[:,'temp_icdsc']
temp.loc[:,'value_char'] = np.nan
temp = temp.drop_duplicates()

# appending temp to data
merged_data = pd.concat([merged_data, temp], axis = 0, ignore_index = True)
merged_data.drop(['temp_icdsc'], axis = 1, inplace = True)
merged_data.sort_values(by = ['MEASUREMENTS_ID'], inplace = True)
merged_data.reset_index(drop = True, inplace = True)
merged_data

,SITE,MRN,ICU_ADMIT,parameter,chartdate,enddate,value_num,value_num_unit,value_char,ADMISSION_ID,PATIENT_ID,MEASUREMENTS_ID
0,FMC ICU,1005018831,2021-08-01 02:14:00,Disorientation,2021-08-01 04:00:00,2021-08-01 04:00:00,0.0,NaN,0.0,A00000,P00033,M00014884
1,FMC ICU,1005018831,2021-08-01 02:14:00,Fluctuations,2021-08-01 04:00:00,2021-08-01 04:00:00,0.0,NaN,0.0,A00000,P00033,M00014884
2,FMC ICU,1005018831,2021-08-01 02:14:00,"Hallucination, Delusion or Psychosis",2021-08-01 04:00:00,2021-08-01 04:00:00,0.0,NaN,0.0,A00000,P00033,M00014884
3,FMC ICU,1005018831,2021-08-01 02:14:00,Altered Level of Consciousness,2021-08-01 04:00:00,2021-08-01 04:00:00,1.0,NaN,1.0,A00000,P00033,M00014884
4,FMC ICU,1005018831,2021-08-01 02:14:00,ICDSC,2021-08-01 04:00:00,2021-08-01 04:00:00,2.0,NaN,2.0,A00000,P00033,M00014884
...,...,...,...,...,...,...,...,...,...,...,...,...
35334,FMC ICU,1021916406,2022-07-22 21:03:00,Altered Level of Consciousness,2022-07-24 04:00:00,2022-07-24 04:00:00,1.0,NaN,1.0,A00351,P00118,M00019110
35335,FMC ICU,1021916406,2022-07-22 21:03:00,Altered Level of Consciousness,2022-07-25 04:00:00,2022-07-25 04:00:00,1.0,NaN,1.0,A00351,P00118,M00019111
35336,FMC ICU,1021916406,2022-07-22 21:03:00,Altered Level of Consciousness,2022-07-25 04:00:00,2022-07-25 04:00:00,1.0,NaN,1.0,A00351,P00118,M00019111
35337,FMC ICU,1021916406,2022-07-22 21:03:00,Altered Level of Consciousness,2022-07-26 05:00:00,2022-07-26 05:00:00,1.0,NaN,1.0,A00351,P00118,M00019112


In [29]:
# preparing temp to append 'ICDSC' to MEASUREMENTS.
merged_data.drop(columns = ['MRN', 'ICU_ADMIT','enddate','value_num_unit','value_char'], inplace = True)
merged_data.rename(columns={'parameter' : 'ITEM_ID',
                     'chartdate' : 'DATETIME',
                     'value_num' : 'VALUE_NUM',}, 
            inplace = True)
merged_data.head()

,SITE,ITEM_ID,DATETIME,VALUE_NUM,ADMISSION_ID,PATIENT_ID,MEASUREMENTS_ID
0,FMC ICU,Disorientation,2021-08-01 04:00:00,0.0,A00000,P00033,M00014884
1,FMC ICU,Fluctuations,2021-08-01 04:00:00,0.0,A00000,P00033,M00014884
2,FMC ICU,"Hallucination, Delusion or Psychosis",2021-08-01 04:00:00,0.0,A00000,P00033,M00014884
3,FMC ICU,Altered Level of Consciousness,2021-08-01 04:00:00,1.0,A00000,P00033,M00014884
4,FMC ICU,ICDSC,2021-08-01 04:00:00,2.0,A00000,P00033,M00014884


In [30]:
# preparing ITEM_ID info
temp = pd.DataFrame(merged_data['ITEM_ID'].unique(), columns = ['LABEL'])
temp['ROW_ID'] = range(count_ref_item, count_ref_item + len(temp))
temp['ITEM_ID'] = temp['ROW_ID'].apply(lambda x: 'I'+str(x).rjust(3, '0'))

# Creating dict using original param labels
item_dict = temp[['LABEL', 'ITEM_ID']].set_index(['LABEL']).to_dict()

# continuing with ITEM_ID info preparation
temp['ABBREVIATION'] = temp['LABEL']
temp['CATEGORY'] = 'ICDSC'
temp['PARAM_TYPE'] = temp['LABEL'].apply(lambda x: 'NUM' if x == 'ICDSC' else 'BIN')
temp['LABEL'].replace({'ICDSC' : 'Intensive Care Delirium Screening Checklist'}, inplace = True)

/tmp/ipykernel_3678869/971497876.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  temp['LABEL'].replace({'ICDSC' : 'Intensive Care Delirium Screening Checklist'}, inplace = True)


In [31]:
# Adjusting data using item_dict
merged_data['LABEL'] = merged_data['ITEM_ID']
merged_data['ITEM_ID'].replace(item_dict['ITEM_ID'], inplace = True)
merged_data.sort_values(by = ['MEASUREMENTS_ID','ITEM_ID'], inplace = True)
merged_data.reset_index(drop = True, inplace = True)

/tmp/ipykernel_3678869/3739936529.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_data['ITEM_ID'].replace(item_dict['ITEM_ID'], inplace = True)


In [32]:
# appending temp to ITEMS
ITEMS_data = pd.concat([ITEMS, temp], axis = 0, ignore_index = True)
MEASUREMENTS_data = pd.concat([MEASUREMENTS,merged_data], axis = 0, ignore_index = True)

/tmp/ipykernel_3678869/2686984526.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  MEASUREMENTS_data = pd.concat([MEASUREMENTS,merged_data], axis = 0, ignore_index = True)


In [33]:
# Trying to free some memory for the following steps
count_ref_measure = int(MEASUREMENTS_data['MEASUREMENTS_ID'].max()[1:]) + 1
count_ref_item = int(ITEMS_data['ITEM_ID'].max()[1:]) + 1
MEASUREMENTS_data.to_pickle('MEASUREMENTS_02.pickle')
ITEMS_data.to_pickle('ITEMS_02.pickle')

In [34]:
del MEASUREMENTS_data
del ITEMS_data
del merged_data
del temp
del item_dict

## 3) ADD PAIN

In [35]:
# preparing PAIN for inclusion
PAIN = pd.read_excel(io = icu_file_path, sheet_name = 'PAIN')
PAIN

,RECORD_ID,SITE,MRN,ICU_ADMIT,parameter,chartdate,value_char
0,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Comment Pain Scale,2021-08-01 08:00:00,when asked if having pain patient knodded his ...
1,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Location of Pain,2021-08-01 08:00:00,Site of Injury
2,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Location of Pain,2021-08-01 16:00:00,Site of Injury
3,190055,FMC ICU,1005018831,2021-08-04 23:56:00,Location of Pain,2021-08-06 20:00:00,Throat
4,190055,FMC ICU,1005018831,2021-08-04 23:56:00,Comment Pain Scale,2021-08-06 20:00:00,Incisional pain denies need for analgesia at...
...,...,...,...,...,...,...,...
5999,209746,FMC ICU,1013229958,2022-07-19 23:23:00,Comment Pain Scale,2022-07-20 20:00:00,Pt. very inconsistant. Initially indicated she...
6000,209746,FMC ICU,1013229958,2022-07-19 23:23:00,Location of Pain,2022-07-21 08:00:00,Headache
6001,209795,FMC ICU,1013129836,2022-07-21 00:22:00,Location of Pain,2022-07-21 20:00:00,Headache
6002,209795,FMC ICU,1013129836,2022-07-21 00:22:00,Comment Pain Scale,2022-07-22 00:10:00,States eyes feel stingy refused eyedrops


In [36]:
# merging data and ADMISSIONS
PAIN = PAIN.merge(right = ADMISSIONS, on = ['RECORD_ID'])
PAIN.drop(['RECORD_ID'], axis = 1, inplace = True)
PAIN.sort_values(by = ['ADMISSION_ID','chartdate'], inplace = True)

In [37]:
PAIN

,SITE,MRN,ICU_ADMIT,parameter,chartdate,value_char,ADMISSION_ID,PATIENT_ID
0,FMC ICU,1005018831,2021-08-01 02:14:00,Comment Pain Scale,2021-08-01 08:00:00,when asked if having pain patient knodded his ...,A00000,P00033
1,FMC ICU,1005018831,2021-08-01 02:14:00,Location of Pain,2021-08-01 08:00:00,Site of Injury,A00000,P00033
2,FMC ICU,1005018831,2021-08-01 02:14:00,Location of Pain,2021-08-01 16:00:00,Site of Injury,A00000,P00033
3,FMC ICU,1005018831,2021-08-04 23:56:00,Location of Pain,2021-08-06 20:00:00,Throat,A00002,P00033
4,FMC ICU,1005018831,2021-08-04 23:56:00,Comment Pain Scale,2021-08-06 20:00:00,Incisional pain denies need for analgesia at...,A00002,P00033
...,...,...,...,...,...,...,...,...
7117,FMC ICU,1013129836,2022-07-21 00:22:00,Location of Pain,2022-07-21 20:00:00,Headache,A00350,P00075
5749,FMC ICU,1013129836,2022-07-21 00:22:00,Comment Pain Scale,2022-07-22 00:10:00,States eyes feel stingy refused eyedrops,A00350,P00075
7119,FMC ICU,1013129836,2022-07-21 00:22:00,Comment Pain Scale,2022-07-22 00:10:00,States eyes feel stingy refused eyedrops,A00350,P00075
5751,FMC ICU,1013129836,2022-07-21 00:22:00,Comment Pain Scale,2022-07-23 00:00:00,Patient too agitated to provide objective answ...,A00350,P00075


In [38]:
# Excluding 'Comment Pain Scale' and 'Location of Pain/Other (Add Comment)'. These info will be added to the comments table.
PAIN = PAIN[(PAIN['parameter'] != 'Comment Pain Scale') & 
            ~((PAIN['parameter'] == 'Location of Pain') & 
            (PAIN['value_char'] == 'Other (Add Comment)'))].reset_index(drop = True)
PAIN

,SITE,MRN,ICU_ADMIT,parameter,chartdate,value_char,ADMISSION_ID,PATIENT_ID
0,FMC ICU,1005018831,2021-08-01 02:14:00,Location of Pain,2021-08-01 08:00:00,Site of Injury,A00000,P00033
1,FMC ICU,1005018831,2021-08-01 02:14:00,Location of Pain,2021-08-01 16:00:00,Site of Injury,A00000,P00033
2,FMC ICU,1005018831,2021-08-04 23:56:00,Location of Pain,2021-08-06 20:00:00,Throat,A00002,P00033
3,FMC ICU,1017255231,2021-08-08 12:16:00,Location of Pain,2021-08-08 16:00:00,Abdomen,A00008,P00108
4,FMC ICU,1017255231,2021-08-08 12:16:00,Pain Scale Patient Goal,2021-08-08 20:00:00,0,A00008,P00108
...,...,...,...,...,...,...,...,...
5468,FMC ICU,1042419836,2022-07-18 16:33:00,Verbal Description of Pain,2022-07-20 20:00:00,Constant,A00347,P00190
5469,FMC ICU,1013229958,2022-07-19 23:23:00,Location of Pain,2022-07-21 08:00:00,Headache,A00348,P00076
5470,FMC ICU,1013229958,2022-07-19 23:23:00,Location of Pain,2022-07-21 08:00:00,Headache,A00348,P00076
5471,FMC ICU,1013129836,2022-07-21 00:22:00,Location of Pain,2022-07-21 20:00:00,Headache,A00350,P00075


In [39]:
# creating MEASUREMENTS_ID based on 'ADMISSION_ID','chartdate'
temp = PAIN[['ADMISSION_ID','chartdate']].groupby(['ADMISSION_ID','chartdate']).count()
temp['MEASUREMENTS_ID'] = range(count_ref_measure, count_ref_measure + len(temp))
temp['MEASUREMENTS_ID'] = temp['MEASUREMENTS_ID'].apply(lambda x: 'M'+str(x).rjust(8, '0'))
temp.reset_index(inplace = True)
PAIN = PAIN.merge(right = temp, on = ['ADMISSION_ID','chartdate'])
PAIN

,SITE,MRN,ICU_ADMIT,parameter,chartdate,value_char,ADMISSION_ID,PATIENT_ID,MEASUREMENTS_ID
0,FMC ICU,1005018831,2021-08-01 02:14:00,Location of Pain,2021-08-01 08:00:00,Site of Injury,A00000,P00033,M00019113
1,FMC ICU,1005018831,2021-08-01 02:14:00,Location of Pain,2021-08-01 16:00:00,Site of Injury,A00000,P00033,M00019114
2,FMC ICU,1005018831,2021-08-04 23:56:00,Location of Pain,2021-08-06 20:00:00,Throat,A00002,P00033,M00019115
3,FMC ICU,1017255231,2021-08-08 12:16:00,Location of Pain,2021-08-08 16:00:00,Abdomen,A00008,P00108,M00019116
4,FMC ICU,1017255231,2021-08-08 12:16:00,Pain Scale Patient Goal,2021-08-08 20:00:00,0,A00008,P00108,M00019117
...,...,...,...,...,...,...,...,...,...
5468,FMC ICU,1042419836,2022-07-18 16:33:00,Verbal Description of Pain,2022-07-20 20:00:00,Constant,A00347,P00190,M00020779
5469,FMC ICU,1013229958,2022-07-19 23:23:00,Location of Pain,2022-07-21 08:00:00,Headache,A00348,P00076,M00020780
5470,FMC ICU,1013229958,2022-07-19 23:23:00,Location of Pain,2022-07-21 08:00:00,Headache,A00348,P00076,M00020780
5471,FMC ICU,1013129836,2022-07-21 00:22:00,Location of Pain,2022-07-21 20:00:00,Headache,A00350,P00075,M00020781


In [40]:
# creating VALUE_NUM
def create_value_num(x):
    try:
        value = float(x)
    except:
        value = None    
    return value
PAIN['value_num'] = PAIN['value_char'].apply(lambda x: create_value_num(x))
PAIN

,SITE,MRN,ICU_ADMIT,parameter,chartdate,value_char,ADMISSION_ID,PATIENT_ID,MEASUREMENTS_ID,value_num
0,FMC ICU,1005018831,2021-08-01 02:14:00,Location of Pain,2021-08-01 08:00:00,Site of Injury,A00000,P00033,M00019113,NaN
1,FMC ICU,1005018831,2021-08-01 02:14:00,Location of Pain,2021-08-01 16:00:00,Site of Injury,A00000,P00033,M00019114,NaN
2,FMC ICU,1005018831,2021-08-04 23:56:00,Location of Pain,2021-08-06 20:00:00,Throat,A00002,P00033,M00019115,NaN
3,FMC ICU,1017255231,2021-08-08 12:16:00,Location of Pain,2021-08-08 16:00:00,Abdomen,A00008,P00108,M00019116,NaN
4,FMC ICU,1017255231,2021-08-08 12:16:00,Pain Scale Patient Goal,2021-08-08 20:00:00,0,A00008,P00108,M00019117,0.0
...,...,...,...,...,...,...,...,...,...,...
5468,FMC ICU,1042419836,2022-07-18 16:33:00,Verbal Description of Pain,2022-07-20 20:00:00,Constant,A00347,P00190,M00020779,NaN
5469,FMC ICU,1013229958,2022-07-19 23:23:00,Location of Pain,2022-07-21 08:00:00,Headache,A00348,P00076,M00020780,NaN
5470,FMC ICU,1013229958,2022-07-19 23:23:00,Location of Pain,2022-07-21 08:00:00,Headache,A00348,P00076,M00020780,NaN
5471,FMC ICU,1013129836,2022-07-21 00:22:00,Location of Pain,2022-07-21 20:00:00,Headache,A00350,P00075,M00020781,NaN


In [41]:
# creating VALUE_CHAR
def create_value_char(x):
    value = np.nan
    try:
        float(x)
    except:
        value = x    
    return value
PAIN['value_char'] = PAIN['value_char'].apply(lambda x: create_value_char(x))
PAIN

,SITE,MRN,ICU_ADMIT,parameter,chartdate,value_char,ADMISSION_ID,PATIENT_ID,MEASUREMENTS_ID,value_num
0,FMC ICU,1005018831,2021-08-01 02:14:00,Location of Pain,2021-08-01 08:00:00,Site of Injury,A00000,P00033,M00019113,NaN
1,FMC ICU,1005018831,2021-08-01 02:14:00,Location of Pain,2021-08-01 16:00:00,Site of Injury,A00000,P00033,M00019114,NaN
2,FMC ICU,1005018831,2021-08-04 23:56:00,Location of Pain,2021-08-06 20:00:00,Throat,A00002,P00033,M00019115,NaN
3,FMC ICU,1017255231,2021-08-08 12:16:00,Location of Pain,2021-08-08 16:00:00,Abdomen,A00008,P00108,M00019116,NaN
4,FMC ICU,1017255231,2021-08-08 12:16:00,Pain Scale Patient Goal,2021-08-08 20:00:00,NaN,A00008,P00108,M00019117,0.0
...,...,...,...,...,...,...,...,...,...,...
5468,FMC ICU,1042419836,2022-07-18 16:33:00,Verbal Description of Pain,2022-07-20 20:00:00,Constant,A00347,P00190,M00020779,NaN
5469,FMC ICU,1013229958,2022-07-19 23:23:00,Location of Pain,2022-07-21 08:00:00,Headache,A00348,P00076,M00020780,NaN
5470,FMC ICU,1013229958,2022-07-19 23:23:00,Location of Pain,2022-07-21 08:00:00,Headache,A00348,P00076,M00020780,NaN
5471,FMC ICU,1013129836,2022-07-21 00:22:00,Location of Pain,2022-07-21 20:00:00,Headache,A00350,P00075,M00020781,NaN


In [42]:
# preparing temp to append 'Pain' to MEASUREMENTS.
PAIN.rename(columns={'parameter' : 'ITEM_ID',
                     'chartdate' : 'DATETIME',
                     'value_num' : 'VALUE_NUM',
                     'value_char' : 'VALUE_CHAR',}, 
            inplace = True)

In [43]:
# preparing ITEM_ID info
temp = pd.DataFrame(PAIN['ITEM_ID'].unique(), columns = ['LABEL'])
temp['ROW_ID'] = range(count_ref_item, count_ref_item + len(temp))
temp['ITEM_ID'] = temp['ROW_ID'].apply(lambda x: 'I'+str(x).rjust(3, '0'))

# Creating dict using original param labels
item_dict = temp[['LABEL', 'ITEM_ID']].set_index(['LABEL']).to_dict()

# continuing with ITEM_ID info preparation
temp['ABBREVIATION'] = temp['LABEL']
temp['CATEGORY'] = 'Pain'
temp['PARAM_TYPE'] = temp['LABEL'].apply(lambda x: 'NUM' if (x == 'Pain Scale at Rest')|
                                                            (x == 'Pain Scale with Activity')|
                                                            (x == 'Pain Scale Patient Goal') else 'CHAR')

In [44]:
# Adjusting data using item_dict
PAIN['LABEL'] = PAIN['ITEM_ID']
PAIN['ITEM_ID'].replace(item_dict['ITEM_ID'], inplace = True)
PAIN.sort_values(by = ['MEASUREMENTS_ID','ITEM_ID'], inplace = True)
PAIN.reset_index(drop = True, inplace = True)
PAIN

/tmp/ipykernel_3678869/3589238282.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  PAIN['ITEM_ID'].replace(item_dict['ITEM_ID'], inplace = True)


,SITE,MRN,ICU_ADMIT,ITEM_ID,DATETIME,VALUE_CHAR,ADMISSION_ID,PATIENT_ID,MEASUREMENTS_ID,VALUE_NUM,LABEL
0,FMC ICU,1005018831,2021-08-01 02:14:00,I016,2021-08-01 08:00:00,Site of Injury,A00000,P00033,M00019113,NaN,Location of Pain
1,FMC ICU,1005018831,2021-08-01 02:14:00,I016,2021-08-01 16:00:00,Site of Injury,A00000,P00033,M00019114,NaN,Location of Pain
2,FMC ICU,1005018831,2021-08-04 23:56:00,I016,2021-08-06 20:00:00,Throat,A00002,P00033,M00019115,NaN,Location of Pain
3,FMC ICU,1017255231,2021-08-08 12:16:00,I016,2021-08-08 16:00:00,Abdomen,A00008,P00108,M00019116,NaN,Location of Pain
4,FMC ICU,1017255231,2021-08-08 12:16:00,I016,2021-08-08 20:00:00,Abdomen,A00008,P00108,M00019117,NaN,Location of Pain
...,...,...,...,...,...,...,...,...,...,...,...
5468,FMC ICU,1042419836,2022-07-18 16:33:00,I019,2022-07-20 20:00:00,Constant,A00347,P00190,M00020779,NaN,Verbal Description of Pain
5469,FMC ICU,1013229958,2022-07-19 23:23:00,I016,2022-07-21 08:00:00,Headache,A00348,P00076,M00020780,NaN,Location of Pain
5470,FMC ICU,1013229958,2022-07-19 23:23:00,I016,2022-07-21 08:00:00,Headache,A00348,P00076,M00020780,NaN,Location of Pain
5471,FMC ICU,1013129836,2022-07-21 00:22:00,I016,2022-07-21 20:00:00,Headache,A00350,P00075,M00020781,NaN,Location of Pain


In [45]:
# appending temp to ITEMS
ITEMS_data = pd.concat([ITEMS, temp], axis = 0, ignore_index = True)
MEASUREMENTS_data = pd.concat([MEASUREMENTS,PAIN], axis = 0, ignore_index = True)
MEASUREMENTS_data

/tmp/ipykernel_3678869/709384095.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  MEASUREMENTS_data = pd.concat([MEASUREMENTS,PAIN], axis = 0, ignore_index = True)


,ROW_ID,MEASUREMENTS_ID,ADMISSION_ID,PATIENT_ID,ITEM_ID,DATETIME,VALUE_NUM,VALUE_UOM,VALUE_CHAR,VALUE_FLAG,SITE,MRN,ICU_ADMIT,LABEL
0,NaN,M00019113,A00000,P00033,I016,2021-08-01 08:00:00,NaN,NaN,Site of Injury,NaN,FMC ICU,1.005019e+09,2021-08-01 02:14:00,Location of Pain
1,NaN,M00019114,A00000,P00033,I016,2021-08-01 16:00:00,NaN,NaN,Site of Injury,NaN,FMC ICU,1.005019e+09,2021-08-01 02:14:00,Location of Pain
2,NaN,M00019115,A00002,P00033,I016,2021-08-06 20:00:00,NaN,NaN,Throat,NaN,FMC ICU,1.005019e+09,2021-08-04 23:56:00,Location of Pain
3,NaN,M00019116,A00008,P00108,I016,2021-08-08 16:00:00,NaN,NaN,Abdomen,NaN,FMC ICU,1.017255e+09,2021-08-08 12:16:00,Location of Pain
4,NaN,M00019117,A00008,P00108,I016,2021-08-08 20:00:00,NaN,NaN,Abdomen,NaN,FMC ICU,1.017255e+09,2021-08-08 12:16:00,Location of Pain
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5468,NaN,M00020779,A00347,P00190,I019,2022-07-20 20:00:00,NaN,NaN,Constant,NaN,FMC ICU,1.042420e+09,2022-07-18 16:33:00,Verbal Description of Pain
5469,NaN,M00020780,A00348,P00076,I016,2022-07-21 08:00:00,NaN,NaN,Headache,NaN,FMC ICU,1.013230e+09,2022-07-19 23:23:00,Location of Pain
5470,NaN,M00020780,A00348,P00076,I016,2022-07-21 08:00:00,NaN,NaN,Headache,NaN,FMC ICU,1.013230e+09,2022-07-19 23:23:00,Location of Pain
5471,NaN,M00020781,A00350,P00075,I016,2022-07-21 20:00:00,NaN,NaN,Headache,NaN,FMC ICU,1.013130e+09,2022-07-21 00:22:00,Location of Pain


In [46]:
# Trying to free some memory for the following steps
count_ref_measure = int(MEASUREMENTS_data['MEASUREMENTS_ID'].max()[1:]) + 1
count_ref_item = int(ITEMS_data['ITEM_ID'].max()[1:]) + 1
MEASUREMENTS_data.to_pickle('MEASUREMENTS_03.pickle')
ITEMS_data.to_pickle('ITEMS_03.pickle')
del MEASUREMENTS_data
del ITEMS_data
del PAIN
del temp
del item_dict

### 4) ADD RASS

In [47]:
# preparing RASS for inclusion
RASS = pd.read_excel(io = icu_file_path, sheet_name = 'RASS')
RASS

,RECORD_ID,SITE,MRN,ICU_ADMIT,parameter,chartdate,value_CHAR,Score
0,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Richmond Agitation Sedation Scale,2021-08-01 04:30:00,Deep Sedation,-4
1,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Richmond Agitation Sedation Scale,2021-08-01 08:00:00,Moderate Sedation,-3
2,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Richmond Agitation Sedation Scale,2021-08-01 12:00:00,Deep Sedation,-4
3,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Richmond Agitation Sedation Scale,2021-08-01 16:00:00,Moderate Sedation,-3
4,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Richmond Agitation Sedation Scale,2021-08-01 20:00:00,Moderate Sedation,-3
...,...,...,...,...,...,...,...,...
10044,207475,FMC ICU,1045503818,2022-05-26 00:34:00,Richmond Agitation Sedation Scale,2022-05-28 15:16:00,Deep Sedation,-4
10045,207475,FMC ICU,1045503818,2022-05-26 00:34:00,Richmond Agitation Sedation Scale,2022-05-28 20:00:00,Unarousable,-5
10046,207475,FMC ICU,1045503818,2022-05-26 00:34:00,Richmond Agitation Sedation Scale,2022-05-29 00:00:00,Unarousable,-5
10047,207475,FMC ICU,1045503818,2022-05-26 00:34:00,Richmond Agitation Sedation Scale,2022-05-29 04:00:00,Unarousable,-5


In [48]:
# merging data and ADMISSIONS
RASS = RASS.merge(right = ADMISSIONS, on = ['RECORD_ID'])
RASS.drop(['RECORD_ID'], axis = 1, inplace = True)
RASS.sort_values(by = ['ADMISSION_ID','chartdate'], inplace = True)

In [49]:
# preparing temp to append 'RASS' to MEASUREMENTS.
RASS.drop(columns = ['value_CHAR'],inplace = True)
RASS.rename(columns={'parameter' : 'ITEM_ID','chartdate' : 'DATETIME','Score' : 'VALUE_NUM',},inplace = True)
RASS['MEASUREMENTS_ID'] = range(count_ref_measure, count_ref_measure + len(RASS))
RASS['MEASUREMENTS_ID'] = RASS['MEASUREMENTS_ID'].apply(lambda x: 'M'+str(x).rjust(8, '0'))

In [50]:
# preparing ITEM_ID info
temp = pd.DataFrame(RASS['ITEM_ID'].unique(), columns = ['LABEL'])
temp['ROW_ID'] = range(count_ref_item, count_ref_item + len(temp))
temp['ITEM_ID'] = temp['ROW_ID'].apply(lambda x: 'I'+str(x).rjust(3, '0'))

# Creating dict using original param labels
item_dict = temp[['LABEL', 'ITEM_ID']].set_index(['LABEL']).to_dict()

# continuing with ITEM_ID info preparation
temp['ABBREVIATION'] = temp['LABEL']
temp['CATEGORY'] = 'RASS'
temp['ABBREVIATION'].replace({'Richmond Agitation Sedation Scale' : 'RASS'}, inplace = True)
temp['PARAM_TYPE'] = 'NUM'

/tmp/ipykernel_3678869/3083518786.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  temp['ABBREVIATION'].replace({'Richmond Agitation Sedation Scale' : 'RASS'}, inplace = True)


In [51]:
# Adjusting data using item_dict
RASS['LABEL'] = RASS['ITEM_ID']
RASS['ITEM_ID'].replace(item_dict['ITEM_ID'], inplace = True)
RASS.sort_values(by = ['MEASUREMENTS_ID','ITEM_ID'], inplace = True)
RASS.reset_index(drop = True, inplace = True)

/tmp/ipykernel_3678869/52300369.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  RASS['ITEM_ID'].replace(item_dict['ITEM_ID'], inplace = True)


In [52]:
# appending
ITEMS_data = pd.concat([ITEMS, temp], axis = 0, ignore_index = True)
MEASUREMENTS_data = pd.concat([MEASUREMENTS,RASS], axis = 0, ignore_index = True)

/tmp/ipykernel_3678869/4144902584.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  MEASUREMENTS_data = pd.concat([MEASUREMENTS,RASS], axis = 0, ignore_index = True)


In [53]:
# Trying to free some memory for the following steps
count_ref_measure = int(MEASUREMENTS_data['MEASUREMENTS_ID'].max()[1:]) + 1
count_ref_item = int(ITEMS_data['ITEM_ID'].max()[1:]) + 1
MEASUREMENTS_data.to_pickle('MEASUREMENTS_04.pickle')
ITEMS_data.to_pickle('ITEMS_04.pickle')
del MEASUREMENTS_data
del ITEMS_data
del RASS
del temp
del item_dict

### 5) Add Urine Output

In [54]:
# preparing Urine for inclusion
Urine = pd.read_excel(io = icu_file_path, sheet_name = 'URINE OUTPUT')
Urine

,RECORD_ID,SITE,MRN,ICU_ADMIT,ParamABB,ChartDate,value_num,Value_NUM_Unit
0,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Urine Indwelling Catheter,2021-08-01 02:16:00,50.0,mL
1,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Urine Indwelling Catheter,2021-08-01 03:00:00,100.0,mL
2,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Urine Indwelling Catheter,2021-08-01 04:00:00,50.0,mL
3,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Urine Indwelling Catheter,2021-08-01 05:00:00,80.0,mL
4,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Urine Indwelling Catheter,2021-08-01 06:00:00,100.0,mL
...,...,...,...,...,...,...,...,...
51015,207475,FMC ICU,1045503818,2022-05-26 00:34:00,Urine Indwelling Catheter,2022-05-29 06:00:00,0.0,mL
51016,207475,FMC ICU,1045503818,2022-05-26 00:34:00,Urine Indwelling Catheter,2022-05-29 07:00:00,0.0,mL
51017,207475,FMC ICU,1045503818,2022-05-26 00:34:00,Urine Indwelling Catheter,2022-05-29 08:00:00,0.0,mL
51018,207475,FMC ICU,1045503818,2022-05-26 00:34:00,Urine Indwelling Catheter,2022-05-29 09:00:00,0.0,mL


In [55]:
# merging Urine output data and ADMISSIONS
Urine = Urine.merge(right = ADMISSIONS, on = ['RECORD_ID'])
Urine.drop(['RECORD_ID'], axis = 1, inplace = True)
Urine.sort_values(by = ['ADMISSION_ID','ChartDate'], inplace = True)
Urine

,SITE,MRN,ICU_ADMIT,ParamABB,ChartDate,value_num,Value_NUM_Unit,ADMISSION_ID,PATIENT_ID
0,FMC ICU,1005018831,2021-08-01 02:14:00,Urine Indwelling Catheter,2021-08-01 02:16:00,50.0,mL,A00000,P00033
1,FMC ICU,1005018831,2021-08-01 02:14:00,Urine Indwelling Catheter,2021-08-01 03:00:00,100.0,mL,A00000,P00033
2,FMC ICU,1005018831,2021-08-01 02:14:00,Urine Indwelling Catheter,2021-08-01 04:00:00,50.0,mL,A00000,P00033
3,FMC ICU,1005018831,2021-08-01 02:14:00,Urine Indwelling Catheter,2021-08-01 05:00:00,80.0,mL,A00000,P00033
4,FMC ICU,1005018831,2021-08-01 02:14:00,Urine Indwelling Catheter,2021-08-01 06:00:00,100.0,mL,A00000,P00033
...,...,...,...,...,...,...,...,...,...
53770,FMC ICU,1021916406,2022-07-22 21:03:00,Urine Indwelling Catheter,2022-07-26 04:00:00,485.0,mL,A00351,P00118
47327,FMC ICU,1021916406,2022-07-22 21:03:00,Urine Indwelling Catheter,2022-07-26 05:00:00,325.0,mL,A00351,P00118
53772,FMC ICU,1021916406,2022-07-22 21:03:00,Urine Indwelling Catheter,2022-07-26 05:00:00,325.0,mL,A00351,P00118
47329,FMC ICU,1021916406,2022-07-22 21:03:00,Urine Indwelling Catheter,2022-07-26 06:00:00,325.0,mL,A00351,P00118


In [56]:
# preparing temp to append 'Urine' to MEASUREMENTS.
Urine.rename(columns={'ParamABB' : 'ITEM_ID',
                     'ChartDate' : 'DATETIME',
                     'value_num' : 'VALUE_NUM',
                     'Value_NUM_Unit' : 'VALUE_UOM',},inplace = True)
Urine['MEASUREMENTS_ID'] = range(count_ref_measure, count_ref_measure + len(Urine))
Urine['MEASUREMENTS_ID'] = Urine['MEASUREMENTS_ID'].apply(lambda x: 'M'+str(x).rjust(8, '0'))
Urine.head()

,SITE,MRN,ICU_ADMIT,ITEM_ID,DATETIME,VALUE_NUM,VALUE_UOM,ADMISSION_ID,PATIENT_ID,MEASUREMENTS_ID
0,FMC ICU,1005018831,2021-08-01 02:14:00,Urine Indwelling Catheter,2021-08-01 02:16:00,50.0,mL,A00000,P00033,M00032161
1,FMC ICU,1005018831,2021-08-01 02:14:00,Urine Indwelling Catheter,2021-08-01 03:00:00,100.0,mL,A00000,P00033,M00032162
2,FMC ICU,1005018831,2021-08-01 02:14:00,Urine Indwelling Catheter,2021-08-01 04:00:00,50.0,mL,A00000,P00033,M00032163
3,FMC ICU,1005018831,2021-08-01 02:14:00,Urine Indwelling Catheter,2021-08-01 05:00:00,80.0,mL,A00000,P00033,M00032164
4,FMC ICU,1005018831,2021-08-01 02:14:00,Urine Indwelling Catheter,2021-08-01 06:00:00,100.0,mL,A00000,P00033,M00032165


In [57]:
# preparing ITEM_ID info
temp = pd.DataFrame(Urine['ITEM_ID'].unique(), columns = ['LABEL'])
temp['ROW_ID'] = range(count_ref_item, count_ref_item + len(temp))
temp['ITEM_ID'] = temp['ROW_ID'].apply(lambda x: 'I'+str(x).rjust(3, '0'))

# Creating dict using original param labels
item_dict = temp[['LABEL', 'ITEM_ID']].set_index(['LABEL']).to_dict()

# continuing with ITEM_ID info preparation
temp['ABBREVIATION'] = temp['LABEL']
temp['CATEGORY'] = 'Urine Volumes'
temp['PARAM_TYPE'] = 'NUM'
temp['UOM'] = 'mL'

In [58]:
# Adjusting data using item_dict
Urine['LABEL'] = Urine['ITEM_ID']
Urine['ITEM_ID'].replace(item_dict['ITEM_ID'], inplace = True)
Urine.sort_values(by = ['MEASUREMENTS_ID','ITEM_ID'], inplace = True)
Urine.reset_index(drop = True, inplace = True)

/tmp/ipykernel_3678869/4276973836.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  Urine['ITEM_ID'].replace(item_dict['ITEM_ID'], inplace = True)


In [59]:
# appending tables
ITEMS_data = pd.concat([ITEMS, temp], axis = 0, ignore_index = True)
MEASUREMENTS_data = pd.concat([MEASUREMENTS,Urine], axis = 0, ignore_index = True)
MEASUREMENTS_data

/tmp/ipykernel_3678869/2718940761.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  MEASUREMENTS_data = pd.concat([MEASUREMENTS,Urine], axis = 0, ignore_index = True)


,ROW_ID,MEASUREMENTS_ID,ADMISSION_ID,PATIENT_ID,ITEM_ID,DATETIME,VALUE_NUM,VALUE_UOM,VALUE_CHAR,VALUE_FLAG,SITE,MRN,ICU_ADMIT,LABEL
0,NaN,M00032161,A00000,P00033,I023,2021-08-01 02:16:00,50.0,mL,NaN,NaN,FMC ICU,1.005019e+09,2021-08-01 02:14:00,Urine Indwelling Catheter
1,NaN,M00032162,A00000,P00033,I023,2021-08-01 03:00:00,100.0,mL,NaN,NaN,FMC ICU,1.005019e+09,2021-08-01 02:14:00,Urine Indwelling Catheter
2,NaN,M00032163,A00000,P00033,I023,2021-08-01 04:00:00,50.0,mL,NaN,NaN,FMC ICU,1.005019e+09,2021-08-01 02:14:00,Urine Indwelling Catheter
3,NaN,M00032164,A00000,P00033,I023,2021-08-01 05:00:00,80.0,mL,NaN,NaN,FMC ICU,1.005019e+09,2021-08-01 02:14:00,Urine Indwelling Catheter
4,NaN,M00032165,A00000,P00033,I023,2021-08-01 06:00:00,100.0,mL,NaN,NaN,FMC ICU,1.005019e+09,2021-08-01 02:14:00,Urine Indwelling Catheter
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64138,NaN,M00096299,A00351,P00118,I023,2022-07-26 04:00:00,485.0,mL,NaN,NaN,FMC ICU,1.021916e+09,2022-07-22 21:03:00,Urine Indwelling Catheter
64139,NaN,M00096300,A00351,P00118,I023,2022-07-26 05:00:00,325.0,mL,NaN,NaN,FMC ICU,1.021916e+09,2022-07-22 21:03:00,Urine Indwelling Catheter
64140,NaN,M00096301,A00351,P00118,I023,2022-07-26 05:00:00,325.0,mL,NaN,NaN,FMC ICU,1.021916e+09,2022-07-22 21:03:00,Urine Indwelling Catheter
64141,NaN,M00096302,A00351,P00118,I023,2022-07-26 06:00:00,325.0,mL,NaN,NaN,FMC ICU,1.021916e+09,2022-07-22 21:03:00,Urine Indwelling Catheter


In [60]:
# Trying to free some memory for the following steps
count_ref_measure = int(MEASUREMENTS_data['MEASUREMENTS_ID'].max()[1:]) + 1
count_ref_item = int(ITEMS_data['ITEM_ID'].max()[1:]) + 1
MEASUREMENTS_data.to_pickle('MEASUREMENTS_05.pickle')
ITEMS_data.to_pickle('ITEMS_05.pickle')
del MEASUREMENTS_data
del ITEMS_data
del Urine
del temp
del item_dict

### 6) ADD Vitals

In [61]:
# preparing Vitals for inclusion
Vitals = pd.read_excel(io = icu_file_path, sheet_name = 'VITALS')
Vitals.head()

,RECORD_ID,SITE,MRN,ICU_ADMIT,CHARTDATE,HR,SPO2,SPO2_#2,SPO2_SITE1,SPO2_SITE2,...,ABP_DIASTOLIC,ABP_MEAN,NIBP_SYSTOLIC,NIBP_DIASTOLIC,NIBP_MEAN,ART_SYSTOLIC_SITE2,ART_DIASTOLIC_SITE2,ART_MEAN_SITE2,MANUAL_SYSTOLIC,MANUAL_DIASTOLIC
0,189844,FMC ICU,1005018831,2021-08-01 02:14:00,2021-08-01 02:15:00,97.0,96.0,NaN,NaN,NaN,...,72.0,95.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,189844,FMC ICU,1005018831,2021-08-01 02:14:00,2021-08-01 02:44:00,NaN,NaN,NaN,95.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,189844,FMC ICU,1005018831,2021-08-01 02:14:00,2021-08-01 03:00:00,94.0,97.0,NaN,NaN,NaN,...,78.0,105.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,189844,FMC ICU,1005018831,2021-08-01 02:14:00,2021-08-01 03:23:00,NaN,NaN,NaN,96.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,189844,FMC ICU,1005018831,2021-08-01 02:14:00,2021-08-01 03:34:00,NaN,NaN,NaN,96.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [62]:
columns_to_pivot = [
    'HR', 'SPO2', 'SPO2_#2', 'SPO2_SITE1', 'SPO2_SITE2', 'ORAL_TEMP',
    'RECTAL_TEMP', 'AXILLARY_TEMP', 'ESOPHAGEAL_TEMP', 'BLADDER_TEMP',
    'PA_CATH_TEMP', 'TYMPANIC_TEMP', 'RESP_RATE_BEDSIDE_MONITOR',
    'RESP_RATE_MANUAL', 'ABP_SYSTOLIC', 'ABP_DIASTOLIC', 'ABP_MEAN',
    'NIBP_SYSTOLIC', 'NIBP_DIASTOLIC', 'NIBP_MEAN', 'ART_SYSTOLIC_SITE2',
    'ART_DIASTOLIC_SITE2', 'ART_MEAN_SITE2', 'MANUAL_SYSTOLIC',
    'MANUAL_DIASTOLIC'
]

new_data = Vitals.melt(
    id_vars=['RECORD_ID', 'CHARTDATE'], 
    value_vars=columns_to_pivot,
    var_name='ITEM_ID', 
    value_name='VALUE_NUM'
)

#new_data = new_data.dropna(subset=['VALUE_NUM'])
new_data

,RECORD_ID,CHARTDATE,ITEM_ID,VALUE_NUM
0,189844,2021-08-01 02:15:00,HR,97.0
1,189844,2021-08-01 02:44:00,HR,NaN
2,189844,2021-08-01 03:00:00,HR,94.0
3,189844,2021-08-01 03:23:00,HR,NaN
4,189844,2021-08-01 03:34:00,HR,NaN
...,...,...,...,...
1720645,207475,2022-05-29 08:00:00,MANUAL_DIASTOLIC,NaN
1720646,207475,2022-05-29 08:11:00,MANUAL_DIASTOLIC,NaN
1720647,207475,2022-05-29 09:00:00,MANUAL_DIASTOLIC,NaN
1720648,207475,2022-05-29 10:00:00,MANUAL_DIASTOLIC,NaN


In [63]:
new_data.rename(columns={'CHARTDATE' : 'DATETIME'}, inplace = True)
new_data['MEASUREMENTS_ID'] = range(count_ref_measure, count_ref_measure + len(new_data))
new_data['MEASUREMENTS_ID'] = new_data['MEASUREMENTS_ID'].apply(lambda x: 'M'+str(x).rjust(8, '0'))
new_data.head()

,RECORD_ID,DATETIME,ITEM_ID,VALUE_NUM,MEASUREMENTS_ID
0,189844,2021-08-01 02:15:00,HR,97.0,M00096304
1,189844,2021-08-01 02:44:00,HR,NaN,M00096305
2,189844,2021-08-01 03:00:00,HR,94.0,M00096306
3,189844,2021-08-01 03:23:00,HR,NaN,M00096307
4,189844,2021-08-01 03:34:00,HR,NaN,M00096308


In [64]:
# preparing ITEM_ID info
temp = pd.DataFrame(new_data['ITEM_ID'].unique(), columns = ['LABEL'])
temp['ROW_ID'] = range(count_ref_item, count_ref_item + len(temp))
temp['ITEM_ID'] = temp['ROW_ID'].apply(lambda x: 'I'+str(x).rjust(3, '0'))

# Creating dict using original param labels
item_dict = temp[['LABEL', 'ITEM_ID']].set_index(['LABEL']).to_dict()

# continuing with ITEM_ID info preparation
temp['ABBREVIATION'] = temp['LABEL']
temp['CATEGORY'] = 'Vitals'
temp['PARAM_TYPE'] = 'NUM'


temp['LABEL'].replace({'HR' : 'Heart Rate',
                       'RESP_RATE_BEDSIDE_MONITOR' : 'Respiratory Rate Bedside Monitor',
                       'ABP_SYSTOLIC' : 'Ambulatory Blood Pressure - Systolic',
                       'ABP_DIASTOLIC' : 'Ambulatory Blood Pressure - Diastolic',
                       'ABP_MEAN' : 'Ambulatory Blood Pressure - Mean',                       
                       'NIBP_SYSTOLIC' : 'Non-Invasive Blood Pressure - Systolic',
                       'NIBP_DIASTOLIC' : 'Non-Invasive Blood Pressure - Diastolic',
                       'NIBP_MEAN' : 'Non-Invasive Blood Pressure - Mean',
                       'SPO2' : 'Oxygen Saturation',
                       'SPO2_SITE1' : 'Oxygen Saturation - Site 1',
                       'SPO2_SITE2' : 'Oxygen Saturation - Site 2',                       
                       'PCO2 Venous' : 'Partial Pressure of Carbon Dioxide - Venous',
                       'ART_SYSTOLIC_SITE2' : 'Arterial Pressure - Systolic',
                       'ART_DIASTOLIC_SITE2' : 'Arterial Pressure - Diastolic',
                       'ART_MEAN_SITE2' : 'Arterial Pressure - Mean',
                       'PA_CATH_TEMP' : 'Pulmonary Artery Catheter Temperature'}, inplace = True)

temp

/tmp/ipykernel_3678869/448494338.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  temp['LABEL'].replace({'HR' : 'Heart Rate',


,LABEL,ROW_ID,ITEM_ID,ABBREVIATION,CATEGORY,PARAM_TYPE
0,Heart Rate,28,I028,HR,Vitals,NUM
1,Oxygen Saturation,29,I029,SPO2,Vitals,NUM
2,SPO2_#2,30,I030,SPO2_#2,Vitals,NUM
3,Oxygen Saturation - Site 1,31,I031,SPO2_SITE1,Vitals,NUM
4,Oxygen Saturation - Site 2,32,I032,SPO2_SITE2,Vitals,NUM
5,ORAL_TEMP,33,I033,ORAL_TEMP,Vitals,NUM
6,RECTAL_TEMP,34,I034,RECTAL_TEMP,Vitals,NUM
7,AXILLARY_TEMP,35,I035,AXILLARY_TEMP,Vitals,NUM
8,ESOPHAGEAL_TEMP,36,I036,ESOPHAGEAL_TEMP,Vitals,NUM
9,BLADDER_TEMP,37,I037,BLADDER_TEMP,Vitals,NUM


In [65]:
# Adjusting data using item_dict
new_data['LABEL'] = new_data['ITEM_ID']
new_data['ITEM_ID'].replace(item_dict['ITEM_ID'], inplace = True)
new_data.sort_values(by = ['MEASUREMENTS_ID','ITEM_ID'], inplace = True)
new_data.reset_index(drop = True, inplace = True)
new_data

/tmp/ipykernel_3678869/353831366.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  new_data['ITEM_ID'].replace(item_dict['ITEM_ID'], inplace = True)


,RECORD_ID,DATETIME,ITEM_ID,VALUE_NUM,MEASUREMENTS_ID,LABEL
0,189844,2021-08-01 02:15:00,I028,97.0,M00096304,HR
1,189844,2021-08-01 02:44:00,I028,NaN,M00096305,HR
2,189844,2021-08-01 03:00:00,I028,94.0,M00096306,HR
3,189844,2021-08-01 03:23:00,I028,NaN,M00096307,HR
4,189844,2021-08-01 03:34:00,I028,NaN,M00096308,HR
...,...,...,...,...,...,...
1720645,207475,2022-05-29 08:00:00,I052,NaN,M01816949,MANUAL_DIASTOLIC
1720646,207475,2022-05-29 08:11:00,I052,NaN,M01816950,MANUAL_DIASTOLIC
1720647,207475,2022-05-29 09:00:00,I052,NaN,M01816951,MANUAL_DIASTOLIC
1720648,207475,2022-05-29 10:00:00,I052,NaN,M01816952,MANUAL_DIASTOLIC


In [66]:
# appending tables
ITEMS_data = pd.concat([ITEMS, temp], axis = 0, ignore_index = True)
MEASUREMENTS_data = pd.concat([MEASUREMENTS,new_data], axis = 0, ignore_index = True)
MEASUREMENTS_data

/tmp/ipykernel_3678869/1703214856.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  MEASUREMENTS_data = pd.concat([MEASUREMENTS,new_data], axis = 0, ignore_index = True)


,ROW_ID,MEASUREMENTS_ID,ADMISSION_ID,PATIENT_ID,ITEM_ID,DATETIME,VALUE_NUM,VALUE_UOM,VALUE_CHAR,VALUE_FLAG,RECORD_ID,LABEL
0,NaN,M00096304,NaN,NaN,I028,2021-08-01 02:15:00,97.0,NaN,NaN,NaN,189844.0,HR
1,NaN,M00096305,NaN,NaN,I028,2021-08-01 02:44:00,NaN,NaN,NaN,NaN,189844.0,HR
2,NaN,M00096306,NaN,NaN,I028,2021-08-01 03:00:00,94.0,NaN,NaN,NaN,189844.0,HR
3,NaN,M00096307,NaN,NaN,I028,2021-08-01 03:23:00,NaN,NaN,NaN,NaN,189844.0,HR
4,NaN,M00096308,NaN,NaN,I028,2021-08-01 03:34:00,NaN,NaN,NaN,NaN,189844.0,HR
...,...,...,...,...,...,...,...,...,...,...,...,...
1720645,NaN,M01816949,NaN,NaN,I052,2022-05-29 08:00:00,NaN,NaN,NaN,NaN,207475.0,MANUAL_DIASTOLIC
1720646,NaN,M01816950,NaN,NaN,I052,2022-05-29 08:11:00,NaN,NaN,NaN,NaN,207475.0,MANUAL_DIASTOLIC
1720647,NaN,M01816951,NaN,NaN,I052,2022-05-29 09:00:00,NaN,NaN,NaN,NaN,207475.0,MANUAL_DIASTOLIC
1720648,NaN,M01816952,NaN,NaN,I052,2022-05-29 10:00:00,NaN,NaN,NaN,NaN,207475.0,MANUAL_DIASTOLIC


In [67]:
# Trying to free some memory for the following steps
count_ref_measure = int(MEASUREMENTS_data['MEASUREMENTS_ID'].max()[1:]) + 1
count_ref_item = int(ITEMS_data['ITEM_ID'].max()[1:]) + 1
MEASUREMENTS_data.to_pickle('MEASUREMENTS_06.pickle')
ITEMS_data.to_pickle('ITEMS_06.pickle')
del MEASUREMENTS_data
del ITEMS_data
del new_data
del temp
del item_dict

### 7) ADD LABS

In [68]:
# preparing LABS for inclusion
data = pd.read_excel(io = icu_file_path, sheet_name = 'LABS')

data.rename(columns={'record_id':'RECORD_ID'}, inplace = True)
data.head()

,RECORD_ID,SITE,MRN,ICU_ADMIT,parameter,ChartDate,validationdate,value_num,Value_NUM_Unit,value_char,Parsed_RefRange_CHAR,Parsed_AbnormalFlags_CHAR
0,189844,FMC ICU,1005018831,2021-08-01 02:14:00,FiO2,2021-08-05 12:15:00,2021-08-05 12:20:00,60.0,%,60,21-100,NaN
1,189844,FMC ICU,1005018831,2021-08-01 02:14:00,FiO2,2021-08-05 05:07:00,2021-08-05 05:16:00,80.0,%,80,21-100,NaN
2,189844,FMC ICU,1005018831,2021-08-01 02:14:00,FiO2,2021-08-05 02:14:00,2021-08-05 02:15:00,100.0,%,100,21-100,NaN
3,189844,FMC ICU,1005018831,2021-08-01 02:14:00,FiO2,2021-08-02 06:49:00,2021-08-02 06:55:00,30.0,%,30,21-100,NaN
4,189844,FMC ICU,1005018831,2021-08-01 02:14:00,FiO2,2021-08-01 19:32:00,2021-08-01 19:40:00,40.0,%,40,21-100,NaN


In [69]:
# merging data and ADMISSIONS
data = data.merge(right = ADMISSIONS, on = ['RECORD_ID'])
data.drop(['RECORD_ID'], axis = 1, inplace = True)
data.sort_values(by = ['ADMISSION_ID','ChartDate'], inplace = True)

In [70]:
# Adjusting Parameters names
data['Value_NUM_Unit'].replace({'x10*12/L' : 'E+12 units/L',
                                'x 10*9/L' : 'E+9 units/L',
                                'mm Hg' : 'mmHg',
                                '10**12/L' : 'E+12 units/L',
                                '10**9/L' : 'E+9 units/L',
                                '_' : np.nan,
                                 'G/L' : 'g/L',
                                'x10*9/L' : 'E+9 units/L',
                                'X10*12/L' : 'E+12 units/L',
                                '10*12/L' : 'E+12 units/L',
                                '10*9/L' : 'E+9 units/L',
                                  
                                '10E12/L' : 'E+12 units/L',
                                '10E9/L' : 'E+9 units/L',
                                  
                                'L/L' : '%'},    
    inplace = True)

/tmp/ipykernel_3678869/4078716091.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['Value_NUM_Unit'].replace({'x10*12/L' : 'E+12 units/L',


In [71]:
# preparing temp to append 'LABS' to MEASUREMENTS.
data.drop(columns = ['validationdate','value_char','Parsed_RefRange_CHAR'],inplace = True)
data.rename(columns={'parameter' : 'ITEM_ID',
                     'ChartDate' : 'DATETIME',
                     'value_num' : 'VALUE_NUM',
                     'Value_NUM_Unit' : 'VALUE_UOM',
                     'Parsed_AbnormalFlags_CHAR' : 'VALUE_FLAG',},inplace = True)
data['MEASUREMENTS_ID'] = range(count_ref_measure, count_ref_measure + len(data))
data['MEASUREMENTS_ID'] = data['MEASUREMENTS_ID'].apply(lambda x: 'M'+str(x).rjust(8, '0'))
data.head()

,SITE,MRN,ICU_ADMIT,ITEM_ID,DATETIME,VALUE_NUM,VALUE_UOM,VALUE_FLAG,ADMISSION_ID,PATIENT_ID,MEASUREMENTS_ID
6,FMC ICU,1005018831,2021-08-01 02:14:00,FiO2,2021-07-31 07:52:00,55.0,%,NaN,A00000,P00033,M01816954
27,FMC ICU,1005018831,2021-08-01 02:14:00,Glucose Blood Gas,2021-07-31 07:52:00,11.1,mmol/L,H,A00000,P00033,M01816955
34,FMC ICU,1005018831,2021-08-01 02:14:00,PCO2 Arterial,2021-07-31 07:52:00,42.0,mmHg,H,A00000,P00033,M01816956
41,FMC ICU,1005018831,2021-08-01 02:14:00,Sodium Blood Gas,2021-07-31 07:52:00,133.0,mmol/L,NaN,A00000,P00033,M01816957
4933,FMC ICU,1005018831,2021-07-31 00:00:00,FiO2,2021-07-31 07:52:00,55.0,%,NaN,A00000,P00033,M01816958


In [72]:
# preparing ITEM_ID info
temp = pd.DataFrame(data['ITEM_ID'].unique(), columns = ['LABEL'])
temp['ROW_ID'] = range(count_ref_item, count_ref_item + len(temp))
temp['ITEM_ID'] = temp['ROW_ID'].apply(lambda x: 'I'+str(x).rjust(3, '0'))

# Creating dict using original param labels
item_dict = temp[['LABEL', 'ITEM_ID']].set_index(['LABEL']).to_dict()

# continuing with ITEM_ID info preparation
temp['ABBREVIATION'] = temp['LABEL']
temp['CATEGORY'] = 'Labs'
temp['PARAM_TYPE'] = 'NUM'
temp['UOM'] = temp.apply(lambda x: 
                         data[data['ITEM_ID'] == x['LABEL']]['VALUE_UOM'].values[0],
                         axis = 1)
temp['LABEL'].replace({'PCO2 Arterial' : 'Partial Pressure of Carbon Dioxide - Arterial',
                       'CO2 Content Blood' : 'Carbon Dioxide Content Blood',
                       'FiO2' : 'Fraction of Inspired Oxygen',
                       'O2 Gradient Arterial OR Capillary' : 'Oxygen Gradient Arterial OR Capillary',                       
                       'PH Arterial' : 'Potential of Hydrogen - Arterial',
                       'PO2 Arterial' : 'Partial Pressure of Oxygen - Arterial',
                       'RBC' : 'Red Blood Cells',
                       'WBC' : 'White Blood Cells',
                       'ALT' : 'Alanine Transaminase',
                       'GGT' : 'Gamma-Glutamyl Transferase',                       
                       'HCO3 Calculated Venous' : 'Bicarbonate Calculated - Venous',
                       'PCO2 Venous' : 'Partial Pressure of Carbon Dioxide - Venous',
                       'AST' : 'Aspartate Aminotransferase',
                       'HCO3 Calculated Manual Entry' : 'Bicarbonate Calculated - Manual Entry',
                       'BNP' : 'Brain Natriuretic Peptide',
                       'CRP' : 'C-Reactive Protein',
                       'Alveolar Arterial O2' : 'Alveolar Arterial Oxygen',
                       'HCO3 Calculated Capillary' : 'Bicarbonate Calculated - Capillary'}, inplace = True)

/tmp/ipykernel_3678869/4101598652.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  temp['LABEL'].replace({'PCO2 Arterial' : 'Partial Pressure of Carbon Dioxide - Arterial',


In [73]:
# Adjusting data using item_dict
data['LABEL'] = data['ITEM_ID']
data['ITEM_ID'].replace(item_dict['ITEM_ID'], inplace = True)

/tmp/ipykernel_3678869/370560010.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['ITEM_ID'].replace(item_dict['ITEM_ID'], inplace = True)


In [74]:
# appending tables
ITEMS_data = pd.concat([ITEMS, temp], axis = 0, ignore_index = True)
MEASUREMENTS_data = pd.concat([MEASUREMENTS,data], axis = 0, ignore_index = True)

/tmp/ipykernel_3678869/3306224264.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  MEASUREMENTS_data = pd.concat([MEASUREMENTS,data], axis = 0, ignore_index = True)


In [75]:
# Trying to free some memory for the following steps
count_ref_measure = int(MEASUREMENTS_data['MEASUREMENTS_ID'].max()[1:]) + 1
count_ref_item = int(ITEMS_data['ITEM_ID'].max()[1:]) + 1

MEASUREMENTS_data.to_pickle('MEASUREMENTS_07.pickle')
ITEMS_data.to_pickle('ITEMS_07.pickle')
del MEASUREMENTS_data
del ITEMS_data
del data
del temp
del item_dict

### 8) ADD SOFA

In [76]:
# preparing SOFA for inclusion
data = pd.read_excel(io = icu_file_path, sheet_name = 'SOFA')
data.head()

,RECORD_ID,SITE,MRN,ICU_ADMIT,CHART_DATE,RESP_SOFA,COAGULATION_SOFA,LIVER_SOFA,CARDIO_SOFA,CNS_SOFA,RENAL_SOFA,SOFA_SCORE
0,207984,FMC ICU,1000292019,2022-06-06 21:05:00,2022-06-07,1,1,0,3,4,0,9
1,207984,FMC ICU,1000292019,2022-06-06 21:05:00,2022-06-08,1,2,0,3,4,0,10
2,207984,FMC ICU,1000292019,2022-06-06 21:05:00,2022-06-09,2,2,0,0,4,0,8
3,207984,FMC ICU,1000292019,2022-06-06 21:05:00,2022-06-10,1,2,0,0,3,0,6
4,207984,FMC ICU,1000292019,2022-06-06 21:05:00,2022-06-11,1,2,0,0,3,0,6


In [77]:
# merging data and ADMISSIONS
data = data.merge(right = ADMISSIONS, on = ['RECORD_ID'])
data.drop(['RECORD_ID'], axis = 1, inplace = True)
data.sort_values(by = ['ADMISSION_ID','CHART_DATE'], inplace = True)
# preparing temp to append SOFA to MEASUREMENTS.
data.rename(columns={'CHART_DATE' : 'DATETIME'}, inplace = True)
data['MEASUREMENTS_ID'] = range(count_ref_measure, count_ref_measure + len(data))
data['MEASUREMENTS_ID'] = data['MEASUREMENTS_ID'].apply(lambda x: 'M'+str(x).rjust(8, '0'))
data.head()

,SITE,MRN,ICU_ADMIT,DATETIME,RESP_SOFA,COAGULATION_SOFA,LIVER_SOFA,CARDIO_SOFA,CNS_SOFA,RENAL_SOFA,SOFA_SCORE,ADMISSION_ID,PATIENT_ID,MEASUREMENTS_ID
209,FMC ICU,1005018831,2021-08-01 02:14:00,2021-08-02,3,0,0,0,0,1,4,A00000,P00033,M01909559
210,FMC ICU,1005018831,2021-08-01 02:14:00,2021-08-03,0,0,0,0,0,0,0,A00000,P00033,M01909560
211,FMC ICU,1005018831,2021-08-01 02:14:00,2021-08-04,0,0,0,0,0,3,3,A00000,P00033,M01909561
37,FMC ICU,1001596889,2021-08-03 03:54:00,2021-08-04,3,0,0,0,3,0,6,A00001,P00005,M01909562
2777,FMC ICU,1001596889,2021-08-03 03:54:00,2021-08-04,3,0,0,0,3,0,6,A00001,P00005,M01909563


In [78]:
new_data = pd.DataFrame()
for column in ['RESP_SOFA','COAGULATION_SOFA','LIVER_SOFA','CARDIO_SOFA','CNS_SOFA','RENAL_SOFA','SOFA_SCORE']:
    temp = data[['MEASUREMENTS_ID','ADMISSION_ID','PATIENT_ID','DATETIME',column]].copy()
    temp.rename(columns={column : 'VALUE_NUM'},inplace = True)
    if column == 'RESP_SOFA':
        temp['ITEM_ID'] = 'Respiratory'
    elif column == 'COAGULATION_SOFA':
        temp['ITEM_ID'] = 'Coagulatory'
    elif column == 'LIVER_SOFA':
        temp['ITEM_ID'] = 'Liver'
    elif column == 'CARDIO_SOFA':
        temp['ITEM_ID'] = 'Cardiovascular'
    elif column == 'CNS_SOFA':
        temp['ITEM_ID'] = 'Neurologic'
    elif column == 'RENAL_SOFA':
        temp['ITEM_ID'] = 'Renal'
    elif column == 'SOFA_SCORE':
        temp['ITEM_ID'] = 'SOFA'
    else:
        pass
    new_data = pd.concat([new_data,temp], axis = 0, ignore_index = True)
data = new_data.copy()
del new_data
data

,MEASUREMENTS_ID,ADMISSION_ID,PATIENT_ID,DATETIME,VALUE_NUM,ITEM_ID
0,M01909559,A00000,P00033,2021-08-02,3,Respiratory
1,M01909560,A00000,P00033,2021-08-03,0,Respiratory
2,M01909561,A00000,P00033,2021-08-04,0,Respiratory
3,M01909562,A00001,P00005,2021-08-04,3,Respiratory
4,M01909563,A00001,P00005,2021-08-04,3,Respiratory
...,...,...,...,...,...,...
24950,M01913119,A00351,P00118,2022-07-25,14,SOFA
24951,M01913120,A00351,P00118,2022-07-26,12,SOFA
24952,M01913121,A00351,P00118,2022-07-26,12,SOFA
24953,M01913122,A00351,P00118,2022-07-27,6,SOFA


In [79]:
# preparing ITEM_ID info
temp = pd.DataFrame(data['ITEM_ID'].unique(), columns = ['LABEL'])
temp['ROW_ID'] = range(count_ref_item, count_ref_item + len(temp))
temp['ITEM_ID'] = temp['ROW_ID'].apply(lambda x: 'I'+str(x).rjust(3, '0'))

# Creating dict using original param labels
item_dict = temp[['LABEL', 'ITEM_ID']].set_index(['LABEL']).to_dict()

# continuing with ITEM_ID info preparation
temp['ABBREVIATION'] = temp['LABEL']
temp['CATEGORY'] = 'SOFA'
temp['PARAM_TYPE'] = 'NUM'
temp['UOM'] = np.nan
temp

,LABEL,ROW_ID,ITEM_ID,ABBREVIATION,CATEGORY,PARAM_TYPE,UOM
0,Respiratory,88,I088,Respiratory,SOFA,NUM,NaN
1,Coagulatory,89,I089,Coagulatory,SOFA,NUM,NaN
2,Liver,90,I090,Liver,SOFA,NUM,NaN
3,Cardiovascular,91,I091,Cardiovascular,SOFA,NUM,NaN
4,Neurologic,92,I092,Neurologic,SOFA,NUM,NaN
5,Renal,93,I093,Renal,SOFA,NUM,NaN
6,SOFA,94,I094,SOFA,SOFA,NUM,NaN


In [80]:
# Adjusting data using item_dict
data['LABEL'] = data['ITEM_ID']
data['ITEM_ID'].replace(item_dict['ITEM_ID'], inplace = True)
data

/tmp/ipykernel_3678869/3222260722.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['ITEM_ID'].replace(item_dict['ITEM_ID'], inplace = True)


,MEASUREMENTS_ID,ADMISSION_ID,PATIENT_ID,DATETIME,VALUE_NUM,ITEM_ID,LABEL
0,M01909559,A00000,P00033,2021-08-02,3,I088,Respiratory
1,M01909560,A00000,P00033,2021-08-03,0,I088,Respiratory
2,M01909561,A00000,P00033,2021-08-04,0,I088,Respiratory
3,M01909562,A00001,P00005,2021-08-04,3,I088,Respiratory
4,M01909563,A00001,P00005,2021-08-04,3,I088,Respiratory
...,...,...,...,...,...,...,...
24950,M01913119,A00351,P00118,2022-07-25,14,I094,SOFA
24951,M01913120,A00351,P00118,2022-07-26,12,I094,SOFA
24952,M01913121,A00351,P00118,2022-07-26,12,I094,SOFA
24953,M01913122,A00351,P00118,2022-07-27,6,I094,SOFA


In [81]:
# appending tables
ITEMS_data = pd.concat([ITEMS, temp], axis = 0, ignore_index = True)
MEASUREMENTS_data = pd.concat([MEASUREMENTS,data], axis = 0, ignore_index = True)

/tmp/ipykernel_3678869/3306224264.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  MEASUREMENTS_data = pd.concat([MEASUREMENTS,data], axis = 0, ignore_index = True)


In [82]:
# Trying to free some memory for the following steps
count_ref_measure = int(MEASUREMENTS_data['MEASUREMENTS_ID'].max()[1:]) + 1
count_ref_item = int(ITEMS_data['ITEM_ID'].max()[1:]) + 1

MEASUREMENTS_data.to_pickle('MEASUREMENTS_08.pickle')
ITEMS_data.to_pickle('ITEMS_08.pickle')
del MEASUREMENTS_data
del ITEMS_data
del data
del temp
del item_dict

### 9) ADD APACHE II

In [83]:
data = pd.read_excel(io = icu_file_path, sheet_name = 'APACHE2')
data.head()

,RECORD_ID,SITE,MRN,CHART_DATE,PATIENT_DATE_OF_BIRTH,HOSPITAL_ADMISSION_DATE_TIME,TYPE_OF_ADMISSION,AP2_ICU_ADMISSION_DATE,AP2_ICU_DISCHARGE_DATE,AP2_ICU_LENGTH_OF_STAY,...,APS,APACHE2_SCORE_EXCLUSION,APACHE2_SCORE,APACHE2_COMPLIANCE,MPM_EXCLUSION,MPM,MIN_VENOUS_BICARBONATE,MAX_VENOUS_BICARBONATE,MIN_CAPILLARY_BICARBONATE,MAX_CAPILLARY_BICARBONATE
0,207984,FMC ICU,1000292019,2022-06-06 21:05:00,1943-05-19,2022-06-02 03:27:00,Elective Post-operative,2022-06-06 21:05:00,2022-06-12 14:09:00,137.066667,...,16,NaN,22.0,11,NaN,17.51,NaN,NaN,NaN,NaN
1,208319,FMC ICU,1000545531,2022-06-14 14:04:00,1953-06-29,2022-06-14 14:04:00,Non-operative,2022-06-14 14:04:00,2022-06-16 22:22:00,52.950000,...,33,NaN,38.0,11,NaN,91.86,NaN,NaN,NaN,NaN
2,205418,FMC ICU,1000552495,2022-04-19 07:55:00,1949-12-06,2022-04-19 07:55:00,Non-operative,2022-04-19 07:55:00,2022-04-22 23:24:00,86.083333,...,30,NaN,40.0,12,NaN,85.06,15.0,31.0,NaN,NaN
3,207924,FMC ICU,1000558914,2022-06-05 11:05:00,1947-06-24,2022-06-05 08:37:00,Emergency Post-operative,2022-06-05 11:05:00,2022-06-16 16:03:00,268.966667,...,10,NaN,15.0,10,NaN,20.80,NaN,NaN,NaN,NaN
4,208785,FMC ICU,1001072881,2022-06-26 11:03:00,1975-10-03,2022-06-26 07:53:00,Emergency Post-operative,2022-06-26 11:03:00,2022-07-03 21:28:00,178.416667,...,20,NaN,22.0,11,NaN,34.14,NaN,NaN,NaN,NaN


In [84]:
# merging data and ADMISSIONS
data = data.merge(right = ADMISSIONS, on = ['RECORD_ID'])
data.drop(['RECORD_ID', 'AP2_ACUTE_RENAL_FAILURE', 'AP2_PRIMARY_BURNS', 'AP2_CORONARYARTERY_BYPASSGRAFT', 'AP2_TRANSFERRED_FROM_OTHER_ICU',
           'MIN_TEMPERATURE', 'MAX_TEMPERATURE', 'MIN_MEAN_ARTERIAL_PRESSURE', 'MAX_MEAN_ARTERIAL_PRESSURE', 'MIN_HEART_RATE',
           'MAX_HEART_RATE', 'MIN_RESPIRATORY_RATE', 'MAX_RESPIRATORY_RATE', 'MIN_FIO2', 'MIN_FIO2_AADO2', 'MIN_FIO2_PO2',
           'MAX_FIO2', 'MAX_FIO2_AADO2', 'MAX_FIO2_PO2', 'MIN_PH', 'MAX_PH', 'MIN_SERUM_SODIUM', 'MAX_SERUM_SODIUM', 
           'MIN_SERUM_POTASSIUM', 'MAX_SERUM_BICARBONATE', 'MIN_SERUM_BICARBONATE', 'MAX_SERUM_POTASSIUM', 
           'MIN_SERUM_CREATININE', 'MAX_SERUM_CREATININE', 'MIN_HEMATOCRIT', 'MAX_HEMATOCRIT', 'MIN_WHITE_BLOOD_CELL',
           'MAX_WHITE_BLOOD_CELL', 'MIN_GLASGOW_COMA_SCORE', 'MAX_GLASGOW_COMA_SCORE', 'APACHE2_SCORE_EXCLUSION',
           'MPM_EXCLUSION', 'MIN_VENOUS_BICARBONATE', 'MAX_VENOUS_BICARBONATE', 'MIN_CAPILLARY_BICARBONATE', 
           'MAX_CAPILLARY_BICARBONATE'], axis = 1, inplace = True)
data.sort_values(by = ['ADMISSION_ID','CHART_DATE'], inplace = True)

data

,SITE,MRN,CHART_DATE,PATIENT_DATE_OF_BIRTH,HOSPITAL_ADMISSION_DATE_TIME,TYPE_OF_ADMISSION,AP2_ICU_ADMISSION_DATE,AP2_ICU_DISCHARGE_DATE,AP2_ICU_LENGTH_OF_STAY,AP2_DIAGNOSIS,...,AP2_RENAL,AP2_IMMUNOCOMPROMISED,AGE_SCORE,CHP,APS,APACHE2_SCORE,APACHE2_COMPLIANCE,MPM,ADMISSION_ID,PATIENT_ID
44,FMC ICU,1005018831,2021-08-01 02:14:00,1963-06-25,2021-08-01 02:14:00,Emergency Post-operative,2021-08-01 02:14:00,2021-08-03 14:08:00,59.900000,"Infection/abscess, other surgery for",...,0.0,0.0,3,NaN,9,12.0,11,14.53,A00000,P00033
6,FMC ICU,1001596889,2021-08-03 03:54:00,1949-09-14,2021-08-03 03:54:00,Non-operative,2021-08-03 03:54:00,2021-08-05 20:51:00,64.950000,"Overdose, street drugs (opiates, cocaine, amph...",...,0.0,0.0,5,NaN,19,24.0,9,NaN,A00001,P00005
350,FMC ICU,1001596889,2021-08-03 03:54:00,1949-09-14,2021-08-03 03:54:00,Non-operative,2021-08-03 03:54:00,2021-08-05 20:51:00,64.950000,"Overdose, street drugs (opiates, cocaine, amph...",...,0.0,0.0,5,NaN,19,24.0,9,NaN,A00001,P00005
45,FMC ICU,1005018831,2021-08-04 23:56:00,1963-06-25,2021-08-01 02:14:00,Emergency Post-operative,2021-08-04 23:56:00,2021-08-07 18:05:00,66.150000,"GI abscess/cyst-primary, surgery for (for comp...",...,0.0,0.0,3,NaN,10,13.0,11,16.40,A00002,P00033
226,FMC ICU,1036362778,2021-08-05 06:52:00,1972-05-05,2021-08-05 06:52:00,Non-operative,2021-08-05 06:52:00,2021-08-12 17:23:00,177.183333,Cardiac arrest (with or without respiratory ar...,...,0.0,0.0,2,NaN,17,19.0,10,41.34,A00003,P00175
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
435,FMC ICU,1041163179,2022-07-20 22:52:00,1983-09-02,2022-07-20 22:52:00,Emergency Post-operative,2022-07-20 22:52:00,2022-07-23 07:20:00,51.083333,"Head/multiple trauma, surgery for",...,0.0,0.0,0,NaN,30,30.0,11,62.50,A00349,P00184
96,FMC ICU,1013129836,2022-07-21 00:22:00,1993-10-07,2022-07-21 00:22:00,Non-operative,2022-07-21 00:22:00,2022-07-24 19:15:00,90.883333,"Overdose, street drugs (opiates, cocaine, amph...",...,0.0,0.0,0,5.0,11,16.0,12,NaN,A00350,P00075
381,FMC ICU,1013129836,2022-07-21 00:22:00,1993-10-07,2022-07-21 00:22:00,Non-operative,2022-07-21 00:22:00,2022-07-24 19:15:00,90.883333,"Overdose, street drugs (opiates, cocaine, amph...",...,0.0,0.0,0,5.0,11,16.0,12,NaN,A00350,P00075
151,FMC ICU,1021916406,2022-07-22 21:03:00,1991-07-08,2022-07-22 21:03:00,Non-operative,2022-07-22 21:03:00,2022-07-26 08:26:00,38.883333,Cardiac arrest (with or without respiratory ar...,...,0.0,0.0,0,NaN,33,33.0,11,84.47,A00351,P00118


In [85]:
# Calculating missing CHP when possible
data['CHP'] = data.apply(lambda x:
                          
                         x['CHP']
                         if
                         ~np.isnan(x['CHP'])
                         else
                                                
                         x['APACHE2_SCORE'] - x['APS'] - x['AGE_SCORE']
                         if
                         ~np.isnan(x['APACHE2_SCORE']) &
                         ~np.isnan(x['APS']) &
                         ~np.isnan(x['AGE_SCORE'])
                         else
                         
                         0
                         if
                         (x['AP2_LIVER'] == 0) &
                         (x['AP2_CARDIOVASCULAR'] == 0) &
                         (x['AP2_RESPIRATORY'] == 0) &
                         (x['AP2_RENAL'] == 0) &
                         (x['AP2_IMMUNOCOMPROMISED'] == 0)
                         else
                         
                         5
                         if
                         ((x['AP2_LIVER'] == 1) |
                         (x['AP2_CARDIOVASCULAR'] == 1) |
                         (x['AP2_RESPIRATORY'] == 1) |
                         (x['AP2_RENAL'] == 1) |
                         (x['AP2_IMMUNOCOMPROMISED'] == 1)) &
                         ((x['ADMISSION_TYPE'] == 'NO_OP') |
                         (x['ADMISSION_TYPE'] == 'EMERG'))
                         else
                         
                         2
                         if
                         ((x['AP2_LIVER'] == 1) |
                         (x['AP2_CARDIOVASCULAR'] == 1) |
                         (x['AP2_RESPIRATORY'] == 1) |
                         (x['AP2_RENAL'] == 1) |
                         (x['AP2_IMMUNOCOMPROMISED'] == 1)) &
                         (x['ADMISSION_TYPE'] == 'ELECT')
                         else
                         
                         0
                         if
                         ~np.isnan(x['AP2_LIVER']) &
                         ~np.isnan(x['AP2_CARDIOVASCULAR']) &
                         ~np.isnan(x['AP2_RESPIRATORY']) &
                         ~np.isnan(x['AP2_RENAL']) &
                         ~np.isnan(x['AP2_IMMUNOCOMPROMISED'])
                         else
                         
                         np.nan,
                         
                         axis = 1)

In [86]:
# Calculating missing APACHE2_SCORE when possible
data['APACHE2_SCORE'] = data.apply(lambda x:
                          
                         x['APACHE2_SCORE']
                         if
                         ~np.isnan(x['APACHE2_SCORE'])
                         else
                                                
                         x['CHP'] + x['APS'] + x['AGE_SCORE']
                         if
                         ~np.isnan(x['CHP']) &
                         ~np.isnan(x['APS']) &
                         ~np.isnan(x['AGE_SCORE'])
                         else
                         
                         np.nan,
                         
                         axis = 1)

In [87]:
data.columns

Index(['SITE', 'MRN', 'CHART_DATE', 'PATIENT_DATE_OF_BIRTH',
       'HOSPITAL_ADMISSION_DATE_TIME', 'TYPE_OF_ADMISSION',
       'AP2_ICU_ADMISSION_DATE', 'AP2_ICU_DISCHARGE_DATE',
       'AP2_ICU_LENGTH_OF_STAY', 'AP2_DIAGNOSIS', 'AP2_AGE', 'AP2_LIVER',
       'AP2_CARDIOVASCULAR', 'AP2_RESPIRATORY', 'AP2_RENAL',
       'AP2_IMMUNOCOMPROMISED', 'AGE_SCORE', 'CHP', 'APS', 'APACHE2_SCORE',
       'APACHE2_COMPLIANCE', 'MPM', 'ADMISSION_ID', 'PATIENT_ID'],
      dtype='object')

In [88]:
# preparing temp to append Apache II to MEASUREMENTS.
data.drop(['AP2_LIVER', 'AP2_CARDIOVASCULAR', 'AP2_RESPIRATORY', 'AP2_RENAL', 'AP2_IMMUNOCOMPROMISED','TYPE_OF_ADMISSION'], 
          axis = 1, inplace = True)
data.rename(columns={'CHART_DATE' : 'DATETIME'}, inplace = True)
data['MEASUREMENTS_ID'] = range(count_ref_measure, count_ref_measure + len(data))
data['MEASUREMENTS_ID'] = data['MEASUREMENTS_ID'].apply(lambda x: 'M'+str(x).rjust(8, '0'))
data.head()

,SITE,MRN,DATETIME,PATIENT_DATE_OF_BIRTH,HOSPITAL_ADMISSION_DATE_TIME,AP2_ICU_ADMISSION_DATE,AP2_ICU_DISCHARGE_DATE,AP2_ICU_LENGTH_OF_STAY,AP2_DIAGNOSIS,AP2_AGE,AGE_SCORE,CHP,APS,APACHE2_SCORE,APACHE2_COMPLIANCE,MPM,ADMISSION_ID,PATIENT_ID,MEASUREMENTS_ID
44,FMC ICU,1005018831,2021-08-01 02:14:00,1963-06-25,2021-08-01 02:14:00,2021-08-01 02:14:00,2021-08-03 14:08:00,59.900000,"Infection/abscess, other surgery for",58.100250,3,0.0,9,12.0,11,14.53,A00000,P00033,M01913124
6,FMC ICU,1001596889,2021-08-03 03:54:00,1949-09-14,2021-08-03 03:54:00,2021-08-03 03:54:00,2021-08-05 20:51:00,64.950000,"Overdose, street drugs (opiates, cocaine, amph...",71.886369,5,0.0,19,24.0,9,NaN,A00001,P00005,M01913125
350,FMC ICU,1001596889,2021-08-03 03:54:00,1949-09-14,2021-08-03 03:54:00,2021-08-03 03:54:00,2021-08-05 20:51:00,64.950000,"Overdose, street drugs (opiates, cocaine, amph...",71.886369,5,0.0,19,24.0,9,NaN,A00001,P00005,M01913126
45,FMC ICU,1005018831,2021-08-04 23:56:00,1963-06-25,2021-08-01 02:14:00,2021-08-04 23:56:00,2021-08-07 18:05:00,66.150000,"GI abscess/cyst-primary, surgery for (for comp...",58.110745,3,0.0,10,13.0,11,16.40,A00002,P00033,M01913127
226,FMC ICU,1036362778,2021-08-05 06:52:00,1972-05-05,2021-08-05 06:52:00,2021-08-05 06:52:00,2021-08-12 17:23:00,177.183333,Cardiac arrest (with or without respiratory ar...,49.250769,2,0.0,17,19.0,10,41.34,A00003,P00175,M01913128


In [89]:
new_data = pd.DataFrame()
for column in ['AGE_SCORE','CHP','APS','APACHE2_SCORE','MPM','APACHE2_COMPLIANCE']:
    temp = data[['MEASUREMENTS_ID','ADMISSION_ID','PATIENT_ID','DATETIME',column]].copy()
    temp.rename(columns={column : 'VALUE_NUM'},inplace = True)
    if column == 'AGE_SCORE':
        temp['ITEM_ID'] = 'AP'
    elif column == 'CHP':
        temp['ITEM_ID'] = 'CHP'
    elif column == 'APS':
        temp['ITEM_ID'] = 'APS'
    elif column == 'APACHE2_SCORE':
        temp['ITEM_ID'] = 'Apache II Score'
    elif column == 'MPM':
        temp['ITEM_ID'] = 'MPM'
    elif column == 'APACHE2_COMPLIANCE':
        temp['ITEM_ID'] = 'Apache II Compliance'
    else:
        pass
    new_data = pd.concat([new_data,temp], axis = 0, ignore_index = True)
new_data = new_data.dropna().reset_index(drop = True)    
data = new_data.copy()
del new_data

In [90]:
# preparing ITEM_ID info
temp = pd.DataFrame(data['ITEM_ID'].unique(), columns = ['LABEL'])
temp['ROW_ID'] = range(count_ref_item, count_ref_item + len(temp))
temp['ITEM_ID'] = temp['ROW_ID'].apply(lambda x: 'I'+str(x).rjust(3, '0'))

# Creating dict using original param labels
item_dict = temp[['LABEL', 'ITEM_ID']].set_index(['LABEL']).to_dict()

# continuing with ITEM_ID info preparation
temp['ABBREVIATION'] = temp['LABEL']
temp['CATEGORY'] = 'Apache II'
temp['PARAM_TYPE'] = 'NUM'
temp['UOM'] = np.nan
temp['LABEL'].replace({'AP' : 'Age Point',
                       'CHP' : 'Chronic Health Point',
                       'APS' : 'Acute Physiological Score',
                       'MPM' : 'Mortality Prediction Model'}, inplace = True)
temp

/tmp/ipykernel_3678869/3380374165.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  temp['LABEL'].replace({'AP' : 'Age Point',


,LABEL,ROW_ID,ITEM_ID,ABBREVIATION,CATEGORY,PARAM_TYPE,UOM
0,Age Point,95,I095,AP,Apache II,NUM,NaN
1,Chronic Health Point,96,I096,CHP,Apache II,NUM,NaN
2,Acute Physiological Score,97,I097,APS,Apache II,NUM,NaN
3,Apache II Score,98,I098,Apache II Score,Apache II,NUM,NaN
4,Mortality Prediction Model,99,I099,MPM,Apache II,NUM,NaN
5,Apache II Compliance,100,I100,Apache II Compliance,Apache II,NUM,NaN


In [91]:
# Adjusting data using item_dict
data['LABEL'] = data['ITEM_ID']
data['ITEM_ID'].replace(item_dict['ITEM_ID'], inplace = True)

/tmp/ipykernel_3678869/370560010.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['ITEM_ID'].replace(item_dict['ITEM_ID'], inplace = True)


In [92]:
# appending tables
ITEMS_data = pd.concat([ITEMS, temp], axis = 0, ignore_index = True)
MEASUREMENTS_data = pd.concat([MEASUREMENTS,data], axis = 0, ignore_index = True)

/tmp/ipykernel_3678869/3306224264.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  MEASUREMENTS_data = pd.concat([MEASUREMENTS,data], axis = 0, ignore_index = True)


In [93]:
# Trying to free some memory for the following steps
count_ref_measure = int(MEASUREMENTS_data['MEASUREMENTS_ID'].max()[1:]) + 1
count_ref_item = int(ITEMS_data['ITEM_ID'].max()[1:]) + 1

MEASUREMENTS_data.to_pickle('MEASUREMENTS_09.pickle')
ITEMS_data.to_pickle('ITEMS_09.pickle')
del MEASUREMENTS_data
del ITEMS_data
del data
del temp
del item_dict

### 10) ADD APACHEE III(IV)

In [94]:
# preparing Apache III(IV) for inclusion
data = pd.read_excel(io = icu_file_path, sheet_name = 'APACHE3')
data.head()

,RECORD_ID,SITE,MRN,ICU_ADMIT,MIN_CHART_DATE_TIME,MAX_CHART_DATE_TIME,MIN_AP4_PULSE,MAX_AP4_PULSE,AP4_PULSE_SCORE,MIN_AP4_MEAN_BLOOD_PRESSURE,...,MAX_AP4_METASTATIC_CANCER,MIN_AP4_LEUK_MULTIPLE_MYELOMA,MAX_AP4_LEUK_MULTIPLE_MYELOMA,MIN_AP4_IMMUNOSUPPRESSION,MAX_AP4_IMMUNOSUPPRESSION,MIN_AP4_CIRRHOSIS,MAX_AP4_CIRRHOSIS,AP4_CHP_SCORE,APACHE4_SCORE,APACHE4_COMPLIANCE
0,207984,FMC ICU,1000292019,2022-06-06 21:05:00,2022-06-06 21:05:00,2022-06-07 21:00:00,56,88,0,70,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,86,16
1,208319,FMC ICU,1000545531,2022-06-14 14:04:00,2022-06-14 14:04:00,2022-06-15 14:00:00,53,99,0,57,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,146,18
2,205418,FMC ICU,1000552495,2022-04-19 07:55:00,2022-04-19 07:55:00,2022-04-20 07:12:00,80,126,7,58,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,10,117,18
3,207924,FMC ICU,1000558914,2022-06-05 11:05:00,2022-06-05 11:05:00,2022-06-06 11:00:00,61,115,5,59,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,11,66,18
4,208785,FMC ICU,1001072881,2022-06-26 11:03:00,2022-06-26 11:03:00,2022-06-27 11:00:00,68,127,7,55,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,89,17


In [95]:
# merging data and ADMISSIONS
data = data.merge(right = ADMISSIONS, on = ['RECORD_ID'])
data.drop(['RECORD_ID', 'MIN_CHART_DATE_TIME', 'MIN_AP4_PULSE', 'MAX_AP4_PULSE', 'AP4_PULSE_SCORE', 
           'MIN_AP4_MEAN_BLOOD_PRESSURE', 'MAX_AP4_MEAN_BLOOD_PRESSURE', 'AP4_MAP_SCORE', 'MIN_AP4_TEMPERATURE', 
           'MAX_AP4_TEMPERATURE', 'AP4_APS_TEMP_SCORE', 'MIN_AP4_VENTILATED_PATIENT', 'MAX_AP4_VENTILATED_PATIENT',
           'MIN_AP4_RESPIRATORY_RATE', 'MAX_AP4_RESPIRATORY_RATE', 'AP4_APS_RESP_SCORE', 'MIN_AP4_INTUBATED_PATIENT', 
           'MAX_AP4_INTUBATED_PATIENT', 'MIN_AP4_FIO2', 'MAX_AP4_FIO2', 'MIN_AP4_PAO2_FIO2_RATIO', 'MAX_AP4_PAO2_FIO2_RATIO',
           'MIN_AP4_PO2_ARTERIAL', 'AP4_APS_OXYGEN_SCORE', 'MIN_AP4_HEMATOCRIT', 'MAX_AP4_HEMATOCRIT', 
           'AP4_APS_HEMATOCRIT_SCORE', 'MIN_AP4_WBC', 'MAX_AP4_WBC', 'AP4_APS_WBC_SCORE', 'MIN_AP4_CREATININE', 
           'MAX_AP4_CREATININE', 'SUM_AP4_CREATININE_URINE', 'MIN_AP4_CHRONIC_DIALYSIS_CHC', 'MAX_AP4_CHRONIC_DIALYSIS_CHC',
           'AP4_APS_CREATININE_SCORE', 'SUM_AP4_URINE_OUTPUT', 'AP4_APS_URINE_SCORE', 'MIN_AP4_BUN', 'MAX_AP4_BUN', 
           'AP4_APS_BUN_SCORE', 'MIN_AP4_SODIUM', 'MAX_AP4_SODIUM', 'AP4_APS_SODIUM_SCORE', 'MIN_AP4_ALBUMIN', 
           'MAX_AP4_ALBUMIN', 'AP4_APS_ALBUMIN_SCORE', 'MIN_AP4_TOTAL_BILIRUBIN', 'MAX_AP4_TOTAL_BILIRUBIN', 
           'AP4_APS_BILIRUBIN_SCORE', 'MIN_AP4_GLUCOSE', 'MAX_AP4_GLUCOSE', 'MV_GLUCOMETER', 'AP4_GLUCOMETER', 
           'AP4_APS_GLUCOSE_SCORE', 'MIN_AP4_PH', 'MAX_AP4_PH', 'MIN_AP4_PCO2', 'MAX_AP4_PCO2', 'AP4_APS_ACID_BASE_SCORE',
           'MIN_AP4_EYE_OPENING_GCS',  'MAX_AP4_EYE_OPENING_GCS', 'MIN_AP4_MOTOR_RESPONSE_GCS', 'MAX_AP4_MOTOR_RESPONSE_GCS',
           'MIN_AP4_VERBAL_RESPONSE_GCS', 'MAX_AP4_VERBAL_RESPONSE_GCS', 'AP4_APS_GCS_SCORE', 'MIN_AP4_RESP_SUPPORT', 
           'MAX_AP4_RESP_SUPPORT', 'MIN_AP4_AGE', 'MAX_AP4_AGE', 'MIN_AP4_AIDS', 'MAX_AP4_AIDS', 'MIN_AP4_HEPATIC_FAILURE',
           'MAX_AP4_HEPATIC_FAILURE', 'MIN_AP4_LYMPHOMA', 'MAX_AP4_LYMPHOMA', 'MIN_AP4_METASTATIC_CANCER', 
           'MAX_AP4_METASTATIC_CANCER', 'MIN_AP4_LEUK_MULTIPLE_MYELOMA', 'MAX_AP4_LEUK_MULTIPLE_MYELOMA', 
           'MIN_AP4_IMMUNOSUPPRESSION', 'MAX_AP4_IMMUNOSUPPRESSION', 'MIN_AP4_CIRRHOSIS', 'MAX_AP4_CIRRHOSIS'], 
          axis = 1, 
          inplace = True)
data.sort_values(by = ['ADMISSION_ID','MAX_CHART_DATE_TIME'], inplace = True)

In [96]:
# preparing temp to append Apache III(IV) to MEASUREMENTS.
data.rename(columns={'MAX_CHART_DATE_TIME' : 'DATETIME'}, inplace = True)
data['MEASUREMENTS_ID'] = range(count_ref_measure, count_ref_measure + len(data))
data['MEASUREMENTS_ID'] = data['MEASUREMENTS_ID'].apply(lambda x: 'M'+str(x).rjust(8, '0'))
data.head()

,SITE,MRN,ICU_ADMIT,DATETIME,AP4_APS_SCORE,AP4_AGE_SCORE,AP4_CHP_SCORE,APACHE4_SCORE,APACHE4_COMPLIANCE,ADMISSION_ID,PATIENT_ID,MEASUREMENTS_ID
44,FMC ICU,1005018831,2021-08-01 02:14:00,2021-08-02 02:00:00,29,5,0,34,18,A00000,P00033,M01913568
6,FMC ICU,1001596889,2021-08-03 03:54:00,2021-08-04 03:26:00,58,16,0,74,13,A00001,P00005,M01913569
350,FMC ICU,1001596889,2021-08-03 03:54:00,2021-08-04 03:26:00,58,16,0,74,13,A00001,P00005,M01913570
45,FMC ICU,1005018831,2021-08-04 23:56:00,2021-08-05 23:52:00,26,5,0,31,15,A00002,P00033,M01913571
226,FMC ICU,1036362778,2021-08-05 06:52:00,2021-08-06 06:00:00,78,5,0,83,18,A00003,P00175,M01913572


In [97]:
new_data = pd.DataFrame()
for column in ['AP4_APS_SCORE', 'AP4_AGE_SCORE', 'AP4_CHP_SCORE', 'APACHE4_SCORE', 'APACHE4_COMPLIANCE']:
    temp = data[['MEASUREMENTS_ID','ADMISSION_ID','PATIENT_ID','DATETIME',column]].copy()
    temp.rename(columns={column : 'VALUE_NUM'},inplace = True)
    if column == 'AP4_APS_SCORE':
        temp['ITEM_ID'] = 'APS'
    elif column == 'AP4_AGE_SCORE':
        temp['ITEM_ID'] = 'AP'
    elif column == 'AP4_CHP_SCORE':
        temp['ITEM_ID'] = 'CHP'
    elif column == 'APACHE4_SCORE':
        temp['ITEM_ID'] = 'Apache III (IV) Score'
    elif column == 'APACHE4_COMPLIANCE':
        temp['ITEM_ID'] = 'Apache III (IV) Compliance'
    else:
        pass
    new_data = pd.concat([new_data,temp], axis = 0, ignore_index = True)
new_data = new_data.dropna().reset_index(drop = True)    
data = new_data.copy()
del new_data

In [98]:
# preparing ITEM_ID info
temp = pd.DataFrame(data['ITEM_ID'].unique(), columns = ['LABEL'])
temp['ROW_ID'] = range(count_ref_item, count_ref_item + len(temp))
temp['ITEM_ID'] = temp['ROW_ID'].apply(lambda x: 'I'+str(x).rjust(3, '0'))

# Creating dict using original param labels
item_dict = temp[['LABEL', 'ITEM_ID']].set_index(['LABEL']).to_dict()

# continuing with ITEM_ID info preparation
temp['ABBREVIATION'] = temp['LABEL']
temp['CATEGORY'] = 'Apache III (IV)'
temp['PARAM_TYPE'] = 'NUM'
temp['UOM'] = np.nan
temp['LABEL'].replace({'AP' : 'Age Point',
                       'CHP' : 'Chronic Health Point',
                       'APS' : 'Acute Physiological Score'}, inplace = True)

/tmp/ipykernel_3678869/1849819350.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  temp['LABEL'].replace({'AP' : 'Age Point',


In [99]:
# Adjusting data using item_dict
data['LABEL'] = data['ITEM_ID']
data['ITEM_ID'].replace(item_dict['ITEM_ID'], inplace = True)

/tmp/ipykernel_3678869/370560010.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['ITEM_ID'].replace(item_dict['ITEM_ID'], inplace = True)


In [100]:
# appending tables
ITEMS_data = pd.concat([ITEMS, temp], axis = 0, ignore_index = True)
MEASUREMENTS_data = pd.concat([MEASUREMENTS,data], axis = 0, ignore_index = True)

/tmp/ipykernel_3678869/3306224264.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  MEASUREMENTS_data = pd.concat([MEASUREMENTS,data], axis = 0, ignore_index = True)


In [101]:
# Trying to free some memory for the following steps
count_ref_item = int(ITEMS_data['ITEM_ID'].max()[1:]) + 1

pickle_out = open("MEASUREMENTS_count_ref_item.pickle","wb")
pickle.dump(count_ref_item, pickle_out)
pickle_out.close()

MEASUREMENTS_data.to_pickle('MEASUREMENTS_10.pickle')
ITEMS_data.to_pickle('ITEMS_10.pickle')
del MEASUREMENTS_data
del ITEMS_data
del data
del temp
del item_dict

In [102]:
MEASUREMENTS_01 = pd.read_pickle('MEASUREMENTS_01.pickle')
MEASUREMENTS_02 = pd.read_pickle('MEASUREMENTS_02.pickle')
MEASUREMENTS_03 = pd.read_pickle('MEASUREMENTS_03.pickle')
MEASUREMENTS_04 = pd.read_pickle('MEASUREMENTS_04.pickle')
MEASUREMENTS_05 = pd.read_pickle('MEASUREMENTS_05.pickle')
MEASUREMENTS_06 = pd.read_pickle('MEASUREMENTS_06.pickle')
MEASUREMENTS_07 = pd.read_pickle('MEASUREMENTS_07.pickle')
MEASUREMENTS_08 = pd.read_pickle('MEASUREMENTS_08.pickle')
MEASUREMENTS_09 = pd.read_pickle('MEASUREMENTS_09.pickle')
MEASUREMENTS_10 = pd.read_pickle('MEASUREMENTS_10.pickle')

MEASUREMENTS = pd.concat([MEASUREMENTS_01,
                          MEASUREMENTS_02,
                          MEASUREMENTS_03,
                          MEASUREMENTS_04,
                          MEASUREMENTS_05,
                          MEASUREMENTS_06,
                          MEASUREMENTS_07,
                          MEASUREMENTS_08,
                          MEASUREMENTS_09,
                          MEASUREMENTS_10
                         ], axis = 0, ignore_index = True)

del MEASUREMENTS_01
del MEASUREMENTS_02
del MEASUREMENTS_03
del MEASUREMENTS_04
del MEASUREMENTS_05
del MEASUREMENTS_06
del MEASUREMENTS_07
del MEASUREMENTS_08
del MEASUREMENTS_09
del MEASUREMENTS_10

MEASUREMENTS['ROW_ID'] = range(0, len(MEASUREMENTS))
MEASUREMENTS.set_index(['ROW_ID'], inplace = True)
MEASUREMENTS

,MEASUREMENTS_ID,ADMISSION_ID,PATIENT_ID,ITEM_ID,DATETIME,VALUE_NUM,VALUE_UOM,VALUE_CHAR,VALUE_FLAG,SITE,MRN,ICU_ADMIT,LABEL,RECORD_ID
ROW_ID,,,,,,,,,,,,,,
0,M00000000,A00000,P00033,I000,2021-08-01 02:14:00,4.0,NaN,NaN,NaN,FMC ICU,1.005019e+09,2021-08-01 02:14:00,Eye Opening,NaN
1,M00000000,A00000,P00033,I001,2021-08-01 02:14:00,5.0,NaN,NaN,NaN,FMC ICU,1.005019e+09,2021-08-01 02:14:00,Verbal Response,NaN
2,M00000000,A00000,P00033,I002,2021-08-01 02:14:00,15.0,NaN,NaN,NaN,FMC ICU,1.005019e+09,2021-08-01 02:14:00,GCS,NaN
3,M00000000,A00000,P00033,I003,2021-08-01 02:14:00,6.0,NaN,NaN,NaN,FMC ICU,1.005019e+09,2021-08-01 02:14:00,Motor Response,NaN
4,M00000001,A00000,P00033,I000,2021-08-01 04:30:00,3.0,NaN,NaN,NaN,FMC ICU,1.005019e+09,2021-08-01 02:14:00,Eye Opening,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2050731,M01914007,A00349,P00184,I105,2022-07-21 22:00:00,15,NaN,NaN,NaN,NaN,NaN,NaT,Apache III (IV) Compliance,NaN
2050732,M01914008,A00350,P00075,I105,2022-07-22 00:10:00,13,NaN,NaN,NaN,NaN,NaN,NaT,Apache III (IV) Compliance,NaN
2050733,M01914009,A00350,P00075,I105,2022-07-22 00:10:00,13,NaN,NaN,NaN,NaN,NaN,NaT,Apache III (IV) Compliance,NaN


In [103]:
MEASUREMENTS.to_csv('MEASUREMENTS.csv', sep = ";", encoding = 'Latin-1')

In [104]:
os.remove('MEASUREMENTS_01.pickle')
os.remove('MEASUREMENTS_02.pickle')
os.remove('MEASUREMENTS_03.pickle')
os.remove('MEASUREMENTS_04.pickle')
os.remove('MEASUREMENTS_05.pickle')
os.remove('MEASUREMENTS_06.pickle')
os.remove('MEASUREMENTS_07.pickle')
os.remove('MEASUREMENTS_08.pickle')
os.remove('MEASUREMENTS_09.pickle')
os.remove('MEASUREMENTS_10.pickle')

In [105]:
ITEMS_01 = pd.read_pickle('ITEMS_01.pickle')
ITEMS_02 = pd.read_pickle('ITEMS_02.pickle')
ITEMS_03 = pd.read_pickle('ITEMS_03.pickle')
ITEMS_04 = pd.read_pickle('ITEMS_04.pickle')
ITEMS_05 = pd.read_pickle('ITEMS_05.pickle')
ITEMS_06 = pd.read_pickle('ITEMS_06.pickle')
ITEMS_07 = pd.read_pickle('ITEMS_07.pickle')
ITEMS_08 = pd.read_pickle('ITEMS_08.pickle')
ITEMS_09 = pd.read_pickle('ITEMS_09.pickle')
ITEMS_10 = pd.read_pickle('ITEMS_10.pickle')

ITEMS = pd.concat([ITEMS_01,
                   ITEMS_02,
                   ITEMS_03,
                   ITEMS_04,
                   ITEMS_05,
                   ITEMS_06,
                   ITEMS_07,
                   ITEMS_08,
                   ITEMS_09,
                   ITEMS_10
                  ], axis = 0, ignore_index = True)

del ITEMS_01
del ITEMS_02
del ITEMS_03
del ITEMS_04
del ITEMS_05
del ITEMS_06
del ITEMS_07
del ITEMS_08
del ITEMS_09
del ITEMS_10

ITEMS.set_index(['ROW_ID'], inplace = True)

ITEMS.to_pickle('ITEMS_MEASUREMENTS.pickle')

os.remove('ITEMS_01.pickle')
os.remove('ITEMS_02.pickle')
os.remove('ITEMS_03.pickle')
os.remove('ITEMS_04.pickle')
os.remove('ITEMS_05.pickle')
os.remove('ITEMS_06.pickle')
os.remove('ITEMS_07.pickle')
os.remove('ITEMS_08.pickle')
os.remove('ITEMS_09.pickle')
os.remove('ITEMS_10.pickle')

In [106]:
ITEMS.to_pickle('ITEMS_MEASUREMENTS.zip')

In [107]:
ITEMS.to_csv('ITEMS_MEASUREMENTS.csv', sep = ";", encoding = 'Latin-1')